<a href="https://colab.research.google.com/github/davimcabral/TE-CNN-AAE/blob/main/TE_CNN_AAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip3 -q install numpy pandas torch umap-learn scikit-learn matplotlib

from __future__ import annotations

import argparse
import hashlib
import json
import math
import re
import shutil
import zipfile
from abc import ABC, abstractmethod
from dataclasses import dataclass, asdict
from itertools import product
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================================
# MAPA CONCEITUAL DO PIPELINE
# ============================================================================
#
# VISÃO GERAL
# -----------
# Este projeto foi dividido em três camadas principais:
#
#   (A) ORIGEM DOS DADOS
#       mockdata / csvdata
#
#   (B) REPRESENTAÇÃO PADRÃO
#       data_79.npz por ticker
#
#   (C) MODELAGEM E SAÍDAS
#       train / embed / labels
#
#
# FLUXO 1 — CONSTRUÇÃO DOS DADOS
# ------------------------------
# Fonte bruta (mock ou CSV)
#        |
#        v
# Daily79Source.build_ticker(ticker)
#        |
#        v
# Daily79Bundle
#   - dates
#   - X_raw
#   - X_norm
#   - eod_close_raw
#   - completeness
#        |
#        v
# save_bundle(...)
#        |
#        v
# out_dir/{TICKER}/data_79.npz
#
#
# FLUXO 2 — TREINAMENTO DO TE-CNN-AAE
# -----------------------------------
# X_norm [N,79]
#    |
#    v
# DataLoader -> batches xb [B,79]
#    |
#    v
# xb.unsqueeze(1) -> [B,1,79]
#    |
#    +------------------------------+
#    |                              |
#    |                              v
#    |                    ytrend_from_intraday_batch(xb)
#    |                              |
#    |                              v
#    |                        y_trend [B,2]
#    |
#    v
# TeCnnAae.forward(xb)
#    |
#    +--> encode(xb)  -> z [B,z_dim]
#    |         |
#    |         +--> td(z)      -> trend_hat [B,2]
#    |         |
#    |         +--> decode(z)  -> x_hat [B,1,79]
#    |
#    v
# perdas:
#   L_recon = MSE(x_hat, xb)
#   L_adv   = MSE(trend_hat, y_trend)
#   L_total = lambda_recon * L_recon + lambda_adv * L_adv
#
#
# FLUXO 3 — EXTRAÇÃO DE EMBEDDINGS
# --------------------------------
# X_norm [N,79]
#    |
#    v
# encoder apenas
#    |
#    v
# z [N,z_dim]
#    |
#    +--> CSV
#    |
#    +--> ARFF (para integração futura com MOA / SMOClust)
#
#
# FLUXO 4 — GERAÇÃO DE LABELS 3-CLASSES
# -------------------------------------
# eod_close_raw [N]
#    |
#    v
# delta = np.diff(eod_close_raw)
#    |
#    v
# find_tau_max_entropy(delta)
#    |
#    v
# tau ótimo
#    |
#    v
# label_3class_from_tau(delta, tau)
#    |
#    v
# labels: Decrease / NoAction / Increase
#
#
# PAPEL DE CADA OBJETO IMPORTANTE
# -------------------------------
# X_raw
#   Série intradiária bruta do dia.
#
# X_norm
#   Série intradiária normalizada em [0,1].
#   É a entrada real do TE-CNN-AAE.
#
# y_trend
#   Rótulo auxiliar de tendência, calculado a partir da própria entrada xb.
#   Não vem do CSV e não vem da reconstrução x_hat.
#
# z
#   Embedding latente aprendido pelo encoder.
#   É o principal produto do modelo para integração futura.
#
# x_hat
#   Reconstrução da série de entrada.
#
# trend_hat
#   Predição do y_trend feita a partir de z.
#
# eod_close_raw
#   Último preço do dia.
#   É usado para gerar labels 3-classes por threshold + entropia.
#
#
# IDEIA CENTRAL DO SCRIPT
# -----------------------
# O script foi desenhado para que "a origem dos dados" seja trocada sem mexer
# no restante do pipeline.
#
# Hoje:
#   mockdata -> data_79.npz -> train/embed/labels
#
# Amanhã:
#   csvdata  -> data_79.npz -> train/embed/labels
#
# Ou seja:
#   train, embed e labels trabalham sempre sobre o MESMO contrato de dados.
# ============================================================================

"""
TE-CNN-AAE pipeline (versão modular e bem documentada)
======================================================

OBJETIVO GERAL
--------------
Este script implementa um pipeline completo para trabalhar com o modelo
TE-CNN-AAE (Trend-Enhanced CNN Adversarial Autoencoder), separando o
problema em 4 camadas:

1) CONSTRUÇÃO DOS DADOS
   - mockdata : cria dados sintéticos com o mesmo formato esperado pelo modelo
   - csvdata  : lê dados reais em CSV e converte para o mesmo formato

2) TREINAMENTO DO MODELO
   - train    : treina o TE-CNN-AAE com os dados já preparados

3) EXTRAÇÃO DE EMBEDDINGS
   - embed    : usa o encoder treinado para transformar cada dia em um vetor z

4) ROTULAGEM 3-CLASSES (OPCIONAL)
   - labels   : gera rótulos Increase / NoAction / Decrease a partir do
                fechamento diário e do threshold por entropia máxima

IDEIA DE PROJETO
----------------
A principal decisão de design deste script é padronizar o formato intermediário
dos dados, para que o restante do pipeline NÃO dependa da origem dos dados.

Ou seja:
- tanto `mockdata` quanto `csvdata` produzem exatamente o mesmo arquivo final;
- depois disso, `train`, `embed` e `labels` funcionam sem saber se os dados
  vieram de um gerador sintético ou de um CSV real.

FORMATO PADRÃO SALVO POR TICKER
-------------------------------
Cada ticker é salvo em:

    out_dir/{TICKER}/data_79.npz

Esse arquivo contém:

- X_raw          : matriz [N,79] com os preços intradiários brutos do dia
- X_norm         : matriz [N,79] com normalização min-max por dia
- dates          : vetor [N] com as datas (YYYY-MM-DD)
- eod_close_raw  : vetor [N] com o último preço do dia (end-of-day)
- completeness   : vetor [N] com a completude da série naquele dia

PAPEL DE CADA REPRESENTAÇÃO
---------------------------
- X_raw:
    Série intradiária "como veio", útil para fechar o preço final do dia e
    para gerar labels 3-classes baseados em delta de fechamento.

- X_norm:
    Série normalizada para [0,1] por dia.
    É essa representação que entra no TE-CNN-AAE, porque o paper descreve
    normalização diária antes da modelagem.

VISÃO DO FLUXO
--------------
mockdata/csvdata  --->  data_79.npz  --->  train  --->  modelo .pt
                                           |
                                           ---> embed ---> embeddings z
                                           |
                                           ---> labels -> rótulos 3 classes

OBSERVAÇÃO IMPORTANTE SOBRE O NOTEBOOK/COLAB
--------------------------------------------
Este script foi escrito para funcionar com `main([...])` no notebook,
em vez de depender de:

    if __name__ == "__main__":
        main()

Isso evita conflito com os argumentos internos do kernel do Jupyter/Colab.
"""



# ============================================================================
# CONSTANTES GLOBAIS
# ============================================================================

# Cada dia intradiário terá 79 pontos.
# Isso corresponde à sessão regular da bolsa com frequência de 5 minutos:
# 390 minutos / 5 + 1 = 79 pontos.
TARGET_LEN = 79

# Lista dos 30 tickers do DJIA usados como referência no projeto.
DJIA_30 = [
    "AAPL", "AMGN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS", "DOW",
    "GS", "HD", "HON", "IBM", "INTC", "JNJ", "JPM", "KO", "MCD", "MMM",
    "MRK", "MSFT", "NKE", "PG", "TRV", "UNH", "V", "VZ", "WBA", "WMT"
]

BUNDLE_DIR = Path("/content/bundles_tecnn")
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)



# ============================================================================
# UTILITÁRIOS GERAIS
# ============================================================================

def stage_outputs(exp_root: Path) -> dict:
    manifest = load_manifest(exp_root)

    analyze_te = (exp_root / "outputs" / "analyze" / "tecnn" / "metrics.json").exists()
    analyze_ae = (exp_root / "outputs" / "analyze" / "cnnae" / "metrics.json").exists()

    te_hsearch_done = (
        (exp_root / "search_autoencoders" / "search_te" / "best_te.json").exists()
        and
        (exp_root / "search_autoencoders" / "search_te" / "te_best" / "best_te.pt").exists()
    )

    ae_hsearch_done = (
        (exp_root / "search_autoencoders" / "search_ae" / "best_ae.json").exists()
        and
        (exp_root / "search_autoencoders" / "search_ae" / "ae_best" / "best_ae.pt").exists()
    )

    return {
        "mockdata": (exp_root / "data" / "SUMMARY.csv").exists(),
        "te_hsearch": bool(manifest.get("te_hsearch_done", False)) or te_hsearch_done,
        "ae_hsearch": bool(manifest.get("ae_hsearch_done", False)) or ae_hsearch_done,
        "te_train": (exp_root / "models" / "tecnn_aae_mock.pt").exists(),
        "te_embed": (exp_root / "outputs" / "embeddings_tecnn_mock.csv").exists(),
        "ae_train": (exp_root / "models" / "cnn_ae_mock.pt").exists(),
        "ae_embed": (exp_root / "outputs" / "embeddings_cnnae_mock.csv").exists(),
        "labels": (exp_root / "outputs" / "labels_mock.csv").exists(),
        "analyze": bool(manifest.get("analyze_done", False)) or (analyze_te and analyze_ae),
        "lstmstudy": bool(manifest.get("lstmstudy_done", False)),
    }

def ensure_dir(p: Path) -> None:
    """
    Garante que a pasta exista.

    Parameters
    ----------
    p : Path
        Caminho da pasta que deve existir.
    """
    p.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int) -> None:
    """
    Define a semente aleatória nas bibliotecas usadas.

    Isso é importante para reprodutibilidade:
    - geração de dados sintéticos
    - inicialização do modelo
    - embaralhamento de batches

    Parameters
    ----------
    seed : int
        Valor da semente.
    """
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def minmax_01(x: np.ndarray) -> np.ndarray:
    """
    Normaliza um vetor para o intervalo [0,1] usando min-max.

    Fórmula:
        x_norm = (x - min(x)) / (max(x) - min(x))

    Se max == min, retorna vetor de zeros para evitar divisão por zero.

    Parameters
    ----------
    x : np.ndarray
        Vetor 1D com valores reais.

    Returns
    -------
    np.ndarray
        Vetor normalizado em [0,1], com dtype float32.
    """
    mn = float(np.min(x))
    mx = float(np.max(x))
    if mx - mn < 1e-12:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)

# ============================================================================
# UTILITÁRIOS PARA SALVAR E CONTINUAR O EXPERIMENTO DE ONDE PAROU
# ============================================================================


ZIP_STAGE_ORDER = {
    "after_mockdata": 10,

    "after_te_hsearch_partial": 15,
    "after_te_hsearch": 16,
    "after_ae_hsearch_partial": 17,
    "after_ae_hsearch": 18,

    "after_te_train": 20,
    "after_te_embed": 30,
    "after_ae_train": 40,
    "after_ae_embed": 50,
    "after_labels": 60,
    "after_analyze": 70,
    "after_lstmstudy_partial": 80,
    "after_lstmstudy": 90,
}

MANIFEST_BOOL_KEYS = [
    "mockdata_done",
    "te_hsearch_done",
    "ae_hsearch_done",
    "te_train_done",
    "te_embed_done",
    "ae_train_done",
    "ae_embed_done",
    "labels_done",
    "analyze_done",
    "lstmstudy_done",
]

MANIFEST_COMPAT_KEYS = [
    "experiment_id",
    "code_version",
    "mock_generator_version",
    "mock_seed_mode",
    "mock_start_date",
    "mock_n_days",
    "mock_seed",
    "arch_version_te",
    "arch_version_ae",
]



COPY_SUFFIX_RE = re.compile(r"\s+\(\d+\)$")

def canonical_zip_stage_name(zip_path: Path) -> str:
    """
    Normaliza nomes como:
      after_lstmstudy_partial.zip
      after_lstmstudy_partial (1).zip
    para a mesma chave canônica: after_lstmstudy_partial
    """
    stem = zip_path.stem.strip().lower()
    stem = COPY_SUFFIX_RE.sub("", stem)
    return stem


def cleanup_duplicate_zips(
    search_dirs: List[Path],
    quarantine_dir: Path = Path("/content/_zip_duplicates"),
    dry_run: bool = False,
) -> List[Path]:
    """
    Mantém apenas o ZIP mais recente por nome canônico de etapa.
    Os demais são movidos para quarantine_dir.

    Critério:
    - agrupa por nome canônico
    - mantém o mais recente por mtime
    - em empate, mantém o de nome lexicograficamente maior

    Retorna a lista final de zips sobreviventes.
    """
    all_zips: List[Path] = []
    for d in search_dirs:
        if d.exists():
            all_zips.extend(sorted(d.glob("*.zip")))

    if not all_zips:
        print("[INFO] Nenhum ZIP encontrado para limpeza.")
        return []

    grouped: Dict[str, List[Path]] = {}
    for zp in all_zips:
        key = canonical_zip_stage_name(zp)
        grouped.setdefault(key, []).append(zp)

    survivors: List[Path] = []
    to_quarantine: List[Path] = []

    for key, group in grouped.items():
        group_sorted = sorted(
            group,
            key=lambda p: (p.stat().st_mtime, p.name.lower()),
            reverse=True,
        )
        keep = group_sorted[0]
        survivors.append(keep)

        duplicates = group_sorted[1:]
        if duplicates:
            print(f"[DUP] {key}: mantendo {keep.name}")
            for dup in duplicates:
                print(f"      removendo da restauração: {dup}")
                to_quarantine.append(dup)

    if to_quarantine and not dry_run:
        quarantine_dir.mkdir(parents=True, exist_ok=True)
        for dup in to_quarantine:
            target = quarantine_dir / dup.name

            # evita colisão no nome dentro da quarentena
            if target.exists():
                base = target.stem
                suffix = target.suffix
                i = 1
                while (quarantine_dir / f"{base}__dup{i}{suffix}").exists():
                    i += 1
                target = quarantine_dir / f"{base}__dup{i}{suffix}"

            shutil.move(str(dup), str(target))
            print(f"[MOVE] {dup} -> {target}")

    survivors = sorted(survivors, key=lambda p: (zip_stage_rank(p), p.name.lower()))
    print("[OK] Zips finais para restauração:")
    for zp in survivors:
        print(" -", zp)

    return survivors


def zip_stage_rank(zip_fp: Path) -> int:
    name = zip_fp.name.lower()
    for prefix, rank in ZIP_STAGE_ORDER.items():
        if name.startswith(prefix):
            return rank
    return 0


def read_manifest_from_zip(zip_fp: Path) -> Dict:
    try:
        with zipfile.ZipFile(zip_fp, "r") as zf:
            if "state/manifest.json" not in zf.namelist():
                return {}
            return json.loads(zf.read("state/manifest.json").decode("utf-8"))
    except Exception:
        return {}


def manifests_compatible(base: Dict, other: Dict) -> bool:
    """
    Compatibilidade "suave":
    se uma chave existir nos dois manifests e os valores diferirem, consideramos incompatível.
    Se a chave estiver ausente em um deles, não bloqueia.
    """
    for k in MANIFEST_COMPAT_KEYS:
        if k in base and k in other and base[k] != other[k]:
            return False
    return True


def merge_manifests(manifests: List[Dict]) -> Dict:
    """
    Mescla manifests preservando progresso.
    - flags booleanas: OR lógico
    - demais metadados: mantém o primeiro valor não nulo e exige compatibilidade prévia
    """
    merged: Dict = {}

    for m in manifests:
        if not m:
            continue

        for k, v in m.items():
            if k in MANIFEST_BOOL_KEYS:
                merged[k] = bool(merged.get(k, False) or bool(v))
            else:
                if k not in merged or merged[k] in (None, "", []):
                    merged[k] = v

    for k in MANIFEST_BOOL_KEYS:
        merged.setdefault(k, False)

    return merged


def restore_many_experiment_states(zip_paths: List[Path], exp_root: Path) -> Dict:
    """
    Restaura múltiplos zips compatíveis no mesmo exp_root.
    - limpa exp_root uma única vez
    - extrai todos os arquivos, exceto state/manifest.json
    - ao final salva um manifest mesclado
    """
    zip_paths = [Path(z) for z in zip_paths if Path(z).exists()]
    if not zip_paths:
        raise ValueError("Nenhum zip fornecido para restauração.")

    manifests: List[Dict] = []
    accepted: List[Path] = []

    # Filtra compatibilidade
    base_manifest: Dict = {}
    for zp in zip_paths:
        m = read_manifest_from_zip(zp)

        if not base_manifest and m:
            base_manifest = m
            accepted.append(zp)
            manifests.append(m)
        else:
            if manifests_compatible(base_manifest, m):
                accepted.append(zp)
                manifests.append(m)
            else:
                print(f"[SKIP] ZIP incompatível ignorado: {zp}")

    # Ordena por estágio e, em empate, por nome
    accepted = sorted(accepted, key=lambda p: (zip_stage_rank(p), p.name.lower()))

    print("Zips que serão restaurados em ordem:")
    for zp in accepted:
        print(" -", zp)

    # Limpa a raiz antes da restauração consolidada
    exp_root = Path(exp_root)
    if exp_root.exists():
        shutil.rmtree(exp_root)
    ensure_exp_dirs(exp_root)

    # Extrai todos, pulando manifest.json interno de cada zip
    for zp in accepted:
        with zipfile.ZipFile(zp, "r") as zf:
            for member in zf.infolist():
                if member.is_dir():
                    continue
                if member.filename == "state/manifest.json":
                    continue
                zf.extract(member, exp_root)

    # Salva manifest consolidado
    merged_manifest = merge_manifests(manifests)
    save_manifest(exp_root, merged_manifest)

    return merged_manifest



def save_manifest(exp_root: Path, manifest: dict) -> None:
    exp_root.mkdir(parents=True, exist_ok=True)
    with open(exp_root / "state" / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


def load_manifest(exp_root: Path) -> dict:
    fp = exp_root / "state" / "manifest.json"
    if not fp.exists():
        return {}
    with open(fp, "r", encoding="utf-8") as f:
        return json.load(f)


def ensure_exp_dirs(exp_root: Path) -> None:
    for sub in ["data", "models", "outputs", "state", "bundles"]:
        (exp_root / sub).mkdir(parents=True, exist_ok=True)


def zip_experiment_state(exp_root, out_zip):
    exp_root = Path(exp_root).resolve()
    out_zip = Path(out_zip).resolve()

    # Nunca permitir que o zip final fique dentro da raiz compactada
    if exp_root == out_zip.parent or exp_root in out_zip.parents:
        raise ValueError(
            f"out_zip={out_zip} não pode ficar dentro de exp_root={exp_root}"
        )

    out_zip.parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in exp_root.rglob("*"):
            if not p.is_file():
                continue

            rel = p.relative_to(exp_root)

            # nunca zipar bundles/ nem outros .zip
            if "bundles" in rel.parts:
                continue
            if p.suffix.lower() == ".zip":
                continue
            if "__pycache__" in rel.parts:
                continue

            zf.write(p, arcname=str(rel))


# ============================================================================
# ESTRUTURA PADRÃO DE DADOS (BUNDLE)
# ============================================================================

@dataclass
class Daily79Bundle:
    """
    Estrutura padronizada para armazenar os dados de UM ticker.

    A ideia deste bundle é encapsular tudo que o restante do pipeline precisa.
    Isso isola a origem dos dados: tanto faz se eles vieram do mock ou de CSV.

    Attributes
    ----------
    ticker : str
        Nome do ticker (ex.: "AAPL").

    dates : np.ndarray
        Vetor [N] com as datas dos dias disponíveis.

    X_raw : np.ndarray
        Matriz [N,79] com preços intradiários brutos.

    X_norm : np.ndarray
        Matriz [N,79] com normalização min-max por dia.

    eod_close_raw : np.ndarray
        Vetor [N] com o último preço do dia (end-of-day close).

    completeness : np.ndarray
        Vetor [N] indicando a fração de pontos disponíveis naquele dia.
        No mock normalmente será 1.0 para todos os dias.
    """
    ticker: str
    dates: np.ndarray
    X_raw: np.ndarray
    X_norm: np.ndarray
    eod_close_raw: np.ndarray
    completeness: np.ndarray

    def validate(self) -> None:
        """
        Faz checagens simples de consistência dimensional.

        Este método é útil para detectar cedo erros de construção do dataset,
        como:
        - dia com número errado de pontos
        - vetores com tamanhos inconsistentes
        - datas e séries com comprimentos diferentes
        """
        n = self.X_raw.shape[0]
        assert self.X_raw.shape == (n, TARGET_LEN)
        assert self.X_norm.shape == (n, TARGET_LEN)
        assert self.dates.shape[0] == n
        assert self.eod_close_raw.shape[0] == n
        assert self.completeness.shape[0] == n


def save_bundle(bundle: Daily79Bundle, out_dir: Path) -> None:
    """
    Salva um bundle em disco no formato padrão do projeto.

    Estrutura salva:
        out_dir/{ticker}/data_79.npz

    Parameters
    ----------
    bundle : Daily79Bundle
        Objeto contendo os dados do ticker.

    out_dir : Path
        Diretório raiz onde os dados serão armazenados.
    """
    bundle.validate()
    tdir = out_dir / bundle.ticker
    ensure_dir(tdir)

    np.savez_compressed(
        tdir / "data_79.npz",
        X_raw=bundle.X_raw.astype(np.float32),
        X_norm=bundle.X_norm.astype(np.float32),
        dates=bundle.dates.astype(str),
        eod_close_raw=bundle.eod_close_raw.astype(np.float32),
        completeness=bundle.completeness.astype(np.float32),
    )


def load_all_tickers_npz(
    data_dir: Path,
    tickers: List[str],
    date_start: Optional[str] = None,
    date_end: Optional[str] = None,
) -> Tuple[np.ndarray, List[Dict[str, Any]]]:
    """
    Carrega os arquivos `data_79.npz` de vários tickers, com filtro opcional por data.
    """

    Xs: List[np.ndarray] = []
    meta: List[Dict[str, Any]] = []

    for tkr in tickers:
        fp = data_dir / tkr / "data_79.npz"
        if not fp.exists():
            print(f"[WARN] arquivo não encontrado: {fp}")
            continue

        d = np.load(fp, allow_pickle=True)
        X_norm = d["X_norm"].astype(np.float32)
        dates = d["dates"].astype(str)
        eod = d["eod_close_raw"].astype(np.float64)

        mask = np.ones(len(dates), dtype=bool)
        if date_start:
            mask &= (dates >= str(date_start))
        if date_end:
            mask &= (dates <= str(date_end))

        X_norm = X_norm[mask]
        dates = dates[mask]
        eod = eod[mask]

        if X_norm.shape[0] == 0:
            continue

        Xs.append(X_norm)
        for i in range(X_norm.shape[0]):
            meta.append({
                "ticker": tkr,
                "date": str(dates[i]),
                "eod_close_raw": float(eod[i]),
            })

    if not Xs:
        raise RuntimeError("Nenhum dado carregado. Verifique os filtros de data ou gere csvdata/mockdata antes.")

    X_all = np.concatenate(Xs, axis=0).astype(np.float32)
    return X_all, meta


def load_per_ticker_close_series(data_dir: Path, ticker: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carrega, para um ticker específico, as datas e o fechamento bruto diário.

    Este método é usado pela etapa de rotulagem 3-classes.

    Parameters
    ----------
    data_dir : Path
        Diretório com os arquivos padronizados.

    ticker : str
        Ticker desejado.

    Returns
    -------
    dates : np.ndarray
        Datas disponíveis.

    eod_close_raw : np.ndarray
        Fechamento bruto de cada dia.
    """
    d = np.load(data_dir / ticker / "data_79.npz", allow_pickle=True)
    return d["dates"].astype(str), d["eod_close_raw"].astype(np.float64)


# ============================================================================
# ABSTRAÇÃO DA FONTE DE DADOS
# ============================================================================

class Daily79Source(ABC):
    """
    Interface abstrata para qualquer fonte que saiba produzir um Daily79Bundle.

    A vantagem desta interface é permitir que o pipeline seja extensível:
    - hoje: MockDaily79Source e CsvDaily79Source
    - amanhã: PolygonDaily79Source, JsonDaily79Source, ParquetDaily79Source etc.

    O restante do pipeline não precisa mudar.
    """

    @abstractmethod
    def build_ticker(self, ticker: str) -> Daily79Bundle:
        """
        Constrói o bundle completo de um ticker.

        Parameters
        ----------
        ticker : str
            Nome do ticker.

        Returns
        -------
        Daily79Bundle
            Objeto pronto para ser salvo e consumido pelas próximas etapas.
        """
        pass


# ============================================================================
# CONSTRUÇÃO DOS DADOS A PARTIR DE UMA FONTE
# ============================================================================

def build_from_source(
    source: Daily79Source,
    tickers: List[str],
    out_dir: Path,
    summary_name: str = "SUMMARY.csv",
) -> None:
    """
    Constrói e salva o dataset padronizado a partir de uma fonte qualquer.

    source:
        objeto que sabe gerar um Daily79Bundle para um ticker.
        Pode ser MockDaily79Source, CsvDaily79Source etc.

    tickers:
        lista de tickers que serão processados.

    out_dir:
        diretório raiz onde os arquivos padronizados serão salvos.

    summary_name:
        nome do CSV-resumo gerado ao final.
    """

    # 1) garante que a pasta de saída exista
    ensure_dir(out_dir)

    # 2) lista onde vamos guardar uma linha-resumo por ticker
    rows = []

    # 3) percorre todos os tickers pedidos
    for tkr in tickers:

        # 3.1) pede à fonte que construa os dados daquele ticker
        #      o retorno precisa ser um Daily79Bundle
        bundle = source.build_ticker(tkr)

        # 3.2) salva esse bundle no formato padrão:
        #      out_dir/{ticker}/data_79.npz
        save_bundle(bundle, out_dir)

        # 3.3) registra um resumo do ticker processado
        rows.append({
            "ticker": tkr,
            "n_days": int(bundle.X_raw.shape[0]),
            "source": source.__class__.__name__,
        })

        # 3.4) imprime feedback simples
        print(f"[OK] {tkr}: {bundle.X_raw.shape[0]} dias")

    # 4) ao final, salva um CSV-resumo com todos os tickers
    pd.DataFrame(rows).to_csv(out_dir / summary_name, index=False)

    # 5) informa onde o resumo foi salvo
    print(f"[OK] summary -> {out_dir / summary_name}")


# ============================================================================
# FONTE 1: DADOS SINTÉTICOS (MOCK)
# ============================================================================

class MockDaily79Source(Daily79Source):
    """
    Fonte sintética de dados.

    Esta classe existe para permitir:
    - testar o pipeline completo sem depender de dados reais
    - verificar se o modelo treina sem erro
    - validar exportação de embeddings
    - facilitar integração futura com SMOClust

    A ideia aqui NÃO é reproduzir fielmente o mercado real, mas sim gerar
    séries plausíveis o suficiente para:
    - o autoencoder ter algo para reconstruir
    - o y_trend ter sentido
    - o Trend Discriminator receber supervisão coerente
    """

    def __init__(self, start_date: str = "2018-01-01", n_days: int = 800, seed: int = 123):
        self.start_date = start_date
        self.n_days = n_days
        self.seed = seed

    @staticmethod
    def _generate_intraday_series(
        rng: np.random.Generator,
        length: int = TARGET_LEN,
        regime: str = "up",
        base_price: float = 100.0,
    ) -> np.ndarray:
        """
        Gera UMA série intradiária sintética de 79 pontos.

        A série é composta por:
        - tendência principal (up, down, flat, etc.)
        - componente sazonal intradiária
        - ruído
        - eventualmente um choque abrupto

        Parameters
        ----------
        rng : np.random.Generator
            Gerador aleatório controlado por semente.

        length : int
            Comprimento da série. Aqui, 79.

        regime : str
            Regime dominante da série.
            Exemplos:
                "up", "down", "flat", "up_vol", "down_vol",
                "reversal_up", "reversal_down", "mean_revert"

        base_price : float
            Nível base do ativo naquele dia.

        Returns
        -------
        np.ndarray
            Série bruta [79] em float32.
        """
        t = np.linspace(0.0, 1.0, length)

        # Ruído branco leve: simula pequenas flutuações de mercado
        noise = rng.normal(0.0, 0.15, size=length)

        # Sazonalidade fraca intradiária:
        # isso ajuda a série sintética a não parecer uma reta pura
        seasonal = 0.3 * np.sin(2 * np.pi * t) + 0.15 * np.sin(6 * np.pi * t)

        # Regime principal
        if regime == "up":
            trend = 2.5 * t
        elif regime == "down":
            trend = -2.5 * t
        elif regime == "flat":
            trend = 0.15 * (t - 0.5)
        elif regime == "up_vol":
            trend = 2.2 * t
            noise += rng.normal(0.0, 0.35, size=length)
        elif regime == "down_vol":
            trend = -2.2 * t
            noise += rng.normal(0.0, 0.35, size=length)
        elif regime == "reversal_up":
            trend = np.where(t < 0.5, -1.6 * t, -0.8 + 3.2 * (t - 0.5))
        elif regime == "reversal_down":
            trend = np.where(t < 0.5, 1.6 * t, 0.8 - 3.2 * (t - 0.5))
        elif regime == "mean_revert":
            trend = 1.2 * np.sin(3 * np.pi * t) * np.exp(-1.2 * t)
        else:
            trend = np.zeros_like(t)

        # Choque exógeno opcional:
        # com pequena probabilidade, o nível da série "salta" a partir de um ponto
        shock = np.zeros(length)
        if rng.random() < 0.18:
            k = rng.integers(low=10, high=length - 10)
            amp = rng.normal(0.0, 1.0)
            shock[k:] += amp

        x = base_price + trend + seasonal + shock + noise

        # Evita preços negativos
        x = np.maximum(x, 1.0).astype(np.float32)
        return x

    def build_ticker(self, ticker: str) -> Daily79Bundle:
        """
        Constrói um bundle sintético completo para um ticker.

        Passos:
        1) define dias úteis
        2) escolhe um regime para cada dia
        3) gera a série bruta do dia
        4) normaliza a série
        5) registra o fechamento final do dia

        Returns
        -------
        Daily79Bundle
            Bundle pronto no schema padrão.
        """
        stable_hash = int(hashlib.md5(ticker.encode("utf-8")).hexdigest()[:8], 16)
        seed = self.seed + (stable_hash % 10000)
        rng = np.random.default_rng(seed)

        dates = pd.bdate_range(
            start=self.start_date,
            periods=self.n_days
        ).strftime("%Y-%m-%d").to_numpy()

        regimes = [
            "up", "down", "flat",
            "up_vol", "down_vol",
            "reversal_up", "reversal_down",
            "mean_revert",
        ]
        probs = np.array([0.18, 0.18, 0.16, 0.12, 0.12, 0.08, 0.08, 0.08], dtype=float)
        probs /= probs.sum()

        X_raw, X_norm, eod, comp = [], [], [], []
        base_price = float(rng.uniform(40.0, 250.0))

        for _ in range(self.n_days):
            regime = rng.choice(regimes, p=probs)

            # O preço base evolui lentamente entre dias
            base_price = max(5.0, base_price + rng.normal(0.0, 0.8))

            x_raw = self._generate_intraday_series(
                rng=rng,
                regime=regime,
                base_price=base_price
            )

            X_raw.append(x_raw)
            X_norm.append(minmax_01(x_raw))
            eod.append(float(x_raw[-1]))
            comp.append(1.0)  # no mock, consideramos série completa

        return Daily79Bundle(
            ticker=ticker,
            dates=np.asarray(dates),
            X_raw=np.stack(X_raw).astype(np.float32),
            X_norm=np.stack(X_norm).astype(np.float32),
            eod_close_raw=np.asarray(eod, dtype=np.float32),
            completeness=np.asarray(comp, dtype=np.float32),
        )


# ============================================================================
# FONTE 2: LEITURA DE CSV
# ============================================================================
# reservado para uso futuro

class CsvDaily79Source(Daily79Source):
    """
    Fonte de dados baseada em CSV.

    Esta classe foi desenhada para ser a substituição natural do mock quando
    você receber dados reais do orientador.

    Ela suporta dois formatos:

    FORMATO A (matrix)
    ------------------
    date,x0,x1,...,x78
    2018-01-02, ...
    2018-01-03, ...

    Aqui, cada linha já representa um dia completo.

    FORMATO B (long)
    ----------------
    timestamp,close
    2018-01-02 09:30:00,172.23
    2018-01-02 09:35:00,172.18
    ...

    Aqui, o código precisa:
    - converter timestamp
    - agrupar por dia
    - montar a série de 79 pontos
    - eventualmente reindexar em grade de 5 minutos
    - preencher pequenos buracos com ffill
    """

    def __init__(
        self,
        csv_dir: str,
        date_col: str = "date",
        timestamp_col: str = "timestamp",
        close_col: str = "close",
        completeness_min: float = 0.80,
        session_start_utc: str = "14:30",
        assume_timezone: str = "UTC",
        require_open_quote: bool = True,
    ):
        self.csv_dir = Path(csv_dir)
        self.date_col = date_col
        self.timestamp_col = timestamp_col
        self.close_col = close_col
        self.completeness_min = completeness_min
        self.session_start_utc = session_start_utc
        self.assume_timezone = assume_timezone
        self.require_open_quote = require_open_quote

    def _load_csv(self, ticker: str) -> pd.DataFrame:
        """
        Lê o CSV de um ticker.

        Espera um arquivo no caminho:
            csv_dir/{ticker}.csv
        """
        fp = self.csv_dir / f"{ticker}.csv"
        if not fp.exists():
            raise FileNotFoundError(f"CSV não encontrado: {fp}")
        return pd.read_csv(fp)

    def _is_matrix_format(self, df: pd.DataFrame) -> bool:
        """
        Detecta se o CSV já está no formato matricial:
        colunas x0, x1, ..., x78.
        """
        feat_cols = [f"x{i}" for i in range(TARGET_LEN)]
        return all(c in df.columns for c in feat_cols)

    def _build_from_matrix(self, ticker: str, df: pd.DataFrame) -> Daily79Bundle:
        """
        Constrói bundle a partir de CSV já pronto no formato [date, x0..x78].

        Aqui o trabalho é simples:
        - lê X_raw diretamente
        - normaliza por linha
        - usa o último valor como fechamento do dia
        """
        feat_cols = [f"x{i}" for i in range(TARGET_LEN)]
        X_raw = df[feat_cols].to_numpy(dtype=np.float32)
        X_norm = np.stack([minmax_01(row) for row in X_raw]).astype(np.float32)
        dates = df[self.date_col].astype(str).to_numpy() if self.date_col in df.columns else np.arange(len(df)).astype(str)
        eod = X_raw[:, -1].astype(np.float32)
        comp = np.ones(X_raw.shape[0], dtype=np.float32)

        return Daily79Bundle(
            ticker=ticker,
            dates=dates,
            X_raw=X_raw,
            X_norm=X_norm,
            eod_close_raw=eod,
            completeness=comp,
        )

    def _build_from_long(self, ticker: str, df: pd.DataFrame) -> Daily79Bundle:
        """
        Constrói bundle a partir de CSV long (timestamp, close), ancorando cada dia
        na sessão fixa do artigo: exige a observação de abertura (14:30 UTC) e
        avalia completude na malha fixa de 79 pontos.
        """
        if self.timestamp_col not in df.columns or self.close_col not in df.columns:
            raise ValueError(
                f"CSV de {ticker} precisa ter colunas "
                f"('{self.timestamp_col}', '{self.close_col}') para o formato long."
            )

        work = df[[self.timestamp_col, self.close_col]].copy()
        ts = pd.to_datetime(work[self.timestamp_col], errors="coerce")

        # Normalização para UTC:
        # - se vier timezone-aware, converte para UTC
        # - se vier naive, assume `assume_timezone`
        if getattr(ts.dt, "tz", None) is None:
            ts = ts.dt.tz_localize(self.assume_timezone)
        ts = ts.dt.tz_convert("UTC")

        work[self.timestamp_col] = ts
        work = work.dropna(subset=[self.timestamp_col, self.close_col]).sort_values(self.timestamp_col)

        # Agrupamos por data UTC
        work["trade_date_utc"] = work[self.timestamp_col].dt.strftime("%Y-%m-%d")

        X_raw, X_norm, dates, eod, comp = [], [], [], [], []

        for day, g in work.groupby("trade_date_utc"):
            s = g.set_index(self.timestamp_col)[self.close_col].sort_index()

            # Grade fixa: 79 pontos de 5 em 5 min a partir da abertura exigida
            session_start = pd.Timestamp(f"{day} {self.session_start_utc}", tz="UTC")
            idx = pd.date_range(start=session_start, periods=TARGET_LEN, freq="5min", tz="UTC")

            s2 = s.reindex(idx)
            completeness = float(s2.notna().mean())
            has_open_quote = bool(pd.notna(s2.iloc[0]))

            if self.require_open_quote and not has_open_quote:
                continue
            if completeness < self.completeness_min:
                continue

            s2 = s2.ffill()
            if s2.isna().any():
                continue

            x_raw = s2.to_numpy(dtype=np.float32)
            if x_raw.shape[0] != TARGET_LEN:
                continue

            X_raw.append(x_raw)
            X_norm.append(minmax_01(x_raw))
            dates.append(day)
            eod.append(float(x_raw[-1]))
            comp.append(float(completeness))

        if not X_raw:
            raise RuntimeError(f"Nenhum dia válido de 79 pontos encontrado para {ticker} no CSV.")

        return Daily79Bundle(
            ticker=ticker,
            dates=np.asarray(dates, dtype=str),
            X_raw=np.stack(X_raw).astype(np.float32),
            X_norm=np.stack(X_norm).astype(np.float32),
            eod_close_raw=np.asarray(eod, dtype=np.float32),
            completeness=np.asarray(comp, dtype=np.float32),
        )

    def build_ticker(self, ticker: str) -> Daily79Bundle:
        """
        Decide automaticamente qual estratégia usar:
        - matrix format
        - long format
        """
        df = self._load_csv(ticker)
        if self._is_matrix_format(df):
            return self._build_from_matrix(ticker, df)
        return self._build_from_long(ticker, df)


# ============================================================================
# GERAÇÃO DE y_trend
# ============================================================================

@torch.no_grad()
def ytrend_from_intraday_batch(x: torch.Tensor, angle_clip_deg: float = 1.0) -> torch.Tensor:
    """
    Calcula y_trend a partir da própria série intradiária x.

    x:
        tensor [B,L] ou [B,1,L]
        onde:
            B = batch size
            L = número de pontos no dia (79)

    angle_clip_deg:
        valor máximo absoluto permitido para o ângulo.
        No seu código atual, o padrão é 1.0.
    """

    # 1) Se x veio como [B,1,L], removemos o canal para trabalhar em [B,L].
    #    Ex.: [32,1,79] -> [32,79]
    if x.dim() == 3:
        x = x.squeeze(1)

    # 2) Verificação defensiva: depois do squeeze, x precisa ser [B,L].
    if x.dim() != 2:
        raise ValueError(f"x must be [B,L] or [B,1,L], got {tuple(x.shape)}")

    # 3) Descobre o tamanho do batch (_) e o comprimento temporal (L).
    _, L = x.shape
    device = x.device

    # 4) Cria o eixo do tempo: t = [0, 1, 2, ..., L-1]
    #    Esse vetor será usado para ajustar implicitamente a reta de tendência.
    t = torch.arange(L, device=device, dtype=x.dtype)

    # 5) Calcula a média de t.
    mean_t = t.mean()

    # 6) Calcula a variância de t.
    #    Isso entra no denominador da fórmula da regressão linear.
    var_t = ((t - mean_t) ** 2).sum().clamp_min(1e-12)

    # 7) Calcula a média de x para cada item do batch.
    #    mean_x terá shape [B,1]
    mean_x = x.mean(dim=1, keepdim=True)

    # 8) Calcula a covariância entre t e x para cada série do batch.
    #    Isso equivale ao numerador do coeficiente angular m.
    cov_tx = ((t - mean_t).unsqueeze(0) * (x - mean_x)).sum(dim=1)

    # 9) Calcula a inclinação m da reta ajustada para cada série.
    m = cov_tx / var_t

    # 10) Converte a inclinação em ângulo (graus).
    angle_deg = torch.atan(m) * (180.0 / math.pi)

    # 11) Limita o ângulo ao intervalo [-angle_clip_deg, +angle_clip_deg].
    #     Isso evita valores extremos e mantém o padrão usado no seu pipeline.
    angle_deg = torch.clamp(angle_deg, -angle_clip_deg, angle_clip_deg)

    # 12) Converte o ângulo em “força de alta”.
    #     Se angle_deg > 0, up cresce; se angle_deg <= 0, up = 0.
    up = torch.clamp(angle_deg, min=0.0, max=angle_clip_deg) / angle_clip_deg

    # 13) Converte o ângulo em “força de baixa”.
    #     Se angle_deg < 0, down cresce; se angle_deg >= 0, down = 0.
    down = torch.clamp(-angle_deg, min=0.0, max=angle_clip_deg) / angle_clip_deg

    # 14) Junta as duas componentes num vetor [B,2]:
    #     coluna 0 = up
    #     coluna 1 = down
    return torch.stack([up, down], dim=1)


# ============================================================================
# BLOCO DE MODELO
# ============================================================================

class ConvBlock1D(nn.Module):
    """
    Bloco simples de convolução 1D + ReLU.

    Uso:
    - encoder
    - decoder
    """
    def __init__(self, cin: int, cout: int, k: int = 7):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout, kernel_size=k, padding=k // 2)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.conv(x))


# ============================================================================
# MODELO
# ============================================================================

class TeCnnAae(nn.Module):
    def __init__(self, input_len: int = TARGET_LEN, arch: Optional[AeArchConfig] = None):
        super().__init__()

        if arch is None:
            arch = AeArchConfig(filters=(16, 32, 128), kernels=(7, 7, 7), z_dim=64, dropout=0.1)

        self.input_len = input_len
        self.z_dim = arch.z_dim
        f1, f2, f3 = arch.filters
        k1, k2, k3 = arch.kernels

        self.arch = arch

        # Encoder
        self.enc_conv1 = ConvBlock1D(1, f1, k1)
        self.enc_pool1 = nn.MaxPool1d(2, 2, ceil_mode=True)

        self.enc_conv2 = ConvBlock1D(f1, f2, k2)
        self.enc_pool2 = nn.MaxPool1d(2, 2, ceil_mode=True)

        self.enc_conv3 = ConvBlock1D(f2, f3, k3)

        L1 = (input_len + 1) // 2
        L2 = (L1 + 1) // 2
        self.enc_len = L2

        self.enc_fc = nn.Linear(f3 * self.enc_len, arch.z_dim)

        # TD com dropout na entrada
        self.td_dropout = nn.Dropout(arch.dropout)
        self.td = nn.Sequential(
            nn.Linear(arch.z_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 2),
            nn.Sigmoid(),
        )

        # Decoder com upsampling real via ConvTranspose1d
        self.dec_fc = nn.Linear(arch.z_dim, f3 * self.enc_len)

        # 20 -> 40
        self.dec_tconv1 = nn.ConvTranspose1d(
            in_channels=f3,
            out_channels=f2,
            kernel_size=k3,
            stride=2,
            padding=k3 // 2,
            output_padding=1,
        )

        # 40 -> 80
        self.dec_tconv2 = nn.ConvTranspose1d(
            in_channels=f2,
            out_channels=f1,
            kernel_size=k2,
            stride=2,
            padding=k2 // 2,
            output_padding=1,
        )

        # 80 -> 80
        self.dec_tconv3 = nn.ConvTranspose1d(
            in_channels=f1,
            out_channels=1,
            kernel_size=k1,
            stride=1,
            padding=k1 // 2,
        )

        self.dec_act = nn.Sigmoid()

    def encode(self, x):
        h = self.enc_conv1(x)
        h = self.enc_pool1(h)
        h = self.enc_conv2(h)
        h = self.enc_pool2(h)
        h = self.enc_conv3(h)
        h = h.flatten(1)
        return self.enc_fc(h)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        f3 = self.arch.filters[2]

        # [B, z_dim] -> [B, f3, enc_len]
        h = self.dec_fc(z).view(z.shape[0], f3, self.enc_len)

        # 20 -> 40
        h = self.dec_tconv1(h)

        # 40 -> 80
        h = self.dec_tconv2(h)

        # 80 -> 80
        h = self.dec_tconv3(h)

        # normaliza saída para [0,1]
        h = self.dec_act(h)

        # ajuste fino final: 80 -> 79
        if h.shape[-1] > self.input_len:
            h = h[..., :self.input_len]
        elif h.shape[-1] < self.input_len:
            h = F.pad(h, (0, self.input_len - h.shape[-1]))

        return h

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        x_hat = self.decode(z)
        trend_hat = self.td(self.td_dropout(z))
        return x_hat, trend_hat, z

# ============================================================================
# ABLAÇÃO: CNN-AE (sem Trend Discriminator)
# ============================================================================
#
# Ideia:
# -------
# Este modelo reutiliza a mesma ideia de encoder + decoder do TE-CNN-AAE,
# mas remove completamente o Trend Discriminator.
#
# Consequência:
# -------------
# - a saída é apenas x_hat e z
# - a loss de treino passa a ser somente L_recon = MSE(x_hat, x)
#
# Papel no artigo:
# ----------------
# Essa variante serve como ablação para verificar se o ganho do TE-CNN-AAE
# vem apenas do autoencoder ou se vem realmente do componente de tendência.
# ============================================================================

class CnnAe(nn.Module):
    def __init__(self, input_len: int = TARGET_LEN, arch: Optional[AeArchConfig] = None):
        super().__init__()

        if arch is None:
            arch = AeArchConfig(filters=(16, 64, 128), kernels=(7, 7, 25), z_dim=64, dropout=0.1)

        self.input_len = input_len
        self.z_dim = arch.z_dim
        f1, f2, f3 = arch.filters
        k1, k2, k3 = arch.kernels

        self.arch = arch

        self.enc_dropout = nn.Dropout(arch.dropout)

        self.enc_conv1 = ConvBlock1D(1, f1, k1)
        self.enc_pool1 = nn.MaxPool1d(2, 2, ceil_mode=True)

        self.enc_conv2 = ConvBlock1D(f1, f2, k2)
        self.enc_pool2 = nn.MaxPool1d(2, 2, ceil_mode=True)

        self.enc_conv3 = ConvBlock1D(f2, f3, k3)

        L1 = (input_len + 1) // 2
        L2 = (L1 + 1) // 2
        self.enc_len = L2

        # faltava esta camada
        self.enc_fc = nn.Linear(f3 * self.enc_len, arch.z_dim)

        self.dec_fc = nn.Linear(arch.z_dim, f3 * self.enc_len)

        self.dec_tconv1 = nn.ConvTranspose1d(
            in_channels=f3,
            out_channels=f2,
            kernel_size=k3,
            stride=2,
            padding=k3 // 2,
            output_padding=1,
        )

        self.dec_tconv2 = nn.ConvTranspose1d(
            in_channels=f2,
            out_channels=f1,
            kernel_size=k2,
            stride=2,
            padding=k2 // 2,
            output_padding=1,
        )

        self.dec_tconv3 = nn.ConvTranspose1d(
            in_channels=f1,
            out_channels=1,
            kernel_size=k1,
            stride=1,
            padding=k1 // 2,
        )

        self.dec_act = nn.Sigmoid()

    def encode(self, x):
        h = self.enc_conv1(x)
        h = self.enc_pool1(h)
        h = self.enc_conv2(h)
        h = self.enc_pool2(h)
        h = self.enc_conv3(h)
        h = h.flatten(1)
        h = self.enc_dropout(h)
        return self.enc_fc(h)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        f3 = self.arch.filters[2]

        h = self.dec_fc(z).view(z.shape[0], f3, self.enc_len)
        h = self.dec_tconv1(h)   # 20 -> 40
        h = self.dec_tconv2(h)   # 40 -> 80
        h = self.dec_tconv3(h)   # 80 -> 80
        h = self.dec_act(h)

        if h.shape[-1] > self.input_len:
            h = h[..., :self.input_len]
        elif h.shape[-1] < self.input_len:
            h = F.pad(h, (0, self.input_len - h.shape[-1]))

        return h

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

# ============================================================================
# TREINAMENTO
# ============================================================================
def make_global_date_chronological_split(
    X_norm: np.ndarray,
    meta: List[Dict[str, Any]],
    test_frac: float = 0.10,
    val_frac_within_train: float = 0.10,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]:
    """
    Split cronológico global baseado nas datas reais do meta.

    Regras:
      - todas as amostras do mesmo dia ficam no mesmo split;
      - os últimos dias únicos vão para teste;
      - os últimos dias do bloco de treino vão para validação;
      - o bloco inicial restante vira treino.

    Isso evita que o split dependa da ordem:
        [ticker1 inteiro] + [ticker2 inteiro] + ...
    """
    if len(X_norm) != len(meta):
        raise RuntimeError(
            f"X_norm e meta desalinhados: {len(X_norm)} amostras vs {len(meta)} metadados."
        )

    if len(X_norm) < 3:
        raise RuntimeError("Dados insuficientes para criar train/val/test cronológico.")

    dates = np.asarray([str(m["date"]) for m in meta], dtype=str)

    # como as datas estão em YYYY-MM-DD, ordenar lexicograficamente preserva ordem temporal
    unique_dates = np.asarray(sorted(set(dates.tolist())), dtype=str)

    n_dates = len(unique_dates)
    if n_dates < 3:
        raise RuntimeError("Datas únicas insuficientes para criar train/val/test cronológico.")

    n_test_dates = int(round(test_frac * n_dates))
    n_test_dates = max(1, n_test_dates)
    n_test_dates = min(n_test_dates, n_dates - 2)

    test_dates = unique_dates[-n_test_dates:]
    train_pool_dates = unique_dates[:-n_test_dates]

    n_train_pool_dates = len(train_pool_dates)
    if n_train_pool_dates < 2:
        raise RuntimeError("Bloco de treino insuficiente após separar o teste cronológico.")

    n_val_dates = int(round(val_frac_within_train * n_train_pool_dates))
    n_val_dates = max(1, n_val_dates)
    n_val_dates = min(n_val_dates, n_train_pool_dates - 1)

    val_dates = train_pool_dates[-n_val_dates:]
    train_dates = train_pool_dates[:-n_val_dates]

    train_mask = np.isin(dates, train_dates)
    val_mask = np.isin(dates, val_dates)
    test_mask = np.isin(dates, test_dates)

    if not np.all(train_mask | val_mask | test_mask):
        raise RuntimeError("Há amostras que não foram alocadas em nenhum split.")

    if np.any(train_mask & val_mask) or np.any(train_mask & test_mask) or np.any(val_mask & test_mask):
        raise RuntimeError("Há sobreposição entre os splits cronológicos.")

    X_train = X_norm[train_mask]
    X_val = X_norm[val_mask]
    X_test = X_norm[test_mask]

    split_info = {
        "type": "chronological_by_unique_date_global",
        "test_frac": float(test_frac),
        "val_frac_within_train": float(val_frac_within_train),
        "n_total_samples": int(len(X_norm)),
        "n_total_unique_dates": int(len(unique_dates)),
        "n_train": int(len(X_train)),
        "n_val": int(len(X_val)),
        "n_test_holdout": int(len(X_test)),
        "n_train_unique_dates": int(len(train_dates)),
        "n_val_unique_dates": int(len(val_dates)),
        "n_test_unique_dates": int(len(test_dates)),
        "train_start_date": str(train_dates[0]),
        "train_end_date": str(train_dates[-1]),
        "val_start_date": str(val_dates[0]),
        "val_end_date": str(val_dates[-1]),
        "test_start_date": str(test_dates[0]),
        "test_end_date": str(test_dates[-1]),
    }

    return X_train, X_val, X_test, split_info



@dataclass(frozen=True)
class AeArchConfig:
    filters: Tuple[int, int, int]
    kernels: Tuple[int, int, int]
    z_dim: int = 64
    dropout: float = 0.1

    def as_dict(self) -> Dict[str, Any]:
        return {
            "filters": list(self.filters),
            "kernels": list(self.kernels),
            "z_dim": self.z_dim,
            "dropout": self.dropout,
        }

@dataclass
class TrainConfig:
    """
    Hiperparâmetros do treinamento do TE-CNN-AAE.
    """
    seed: int = 123
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    epochs: int = 20
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 0.0
    dropout: float = 0.1
    z_dim: int = 64
    lambda_recon: float = 1.0
    lambda_adv: float = 1.0
    angle_clip_deg: float = 1.0
    num_workers: int = 0


class XDataset(torch.utils.data.Dataset):
    """
    Dataset simples que expõe apenas X_norm.

    O y_trend não é armazenado no disco separadamente; ele é calculado
    dinamicamente durante o treino, a partir do próprio batch x.
    """
    def __init__(self, X_norm: np.ndarray):
        self.X = np.asarray(X_norm, dtype=np.float32)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        return torch.from_numpy(self.X[idx])


# ============================================================================
# TREINAMENTO
# ============================================================================
# --------------------------------------------------------------------------
# RESUMO DO TREINO
# --------------------------------------------------------------------------
# Entrada do treino:
#   X_norm [N,79]
#
# Em cada batch:
#   xb [B,79] -> [B,1,79]
#   y_trend = função(xb)
#   x_hat, trend_hat, z = modelo(xb)
#   L_recon = MSE(x_hat, xb)
#   L_adv   = MSE(trend_hat, y_trend)
#   L_total = lambda_recon * L_recon + lambda_adv * L_adv
# --------------------------------------------------------------------------
def fit_tecnn_trial(
    X_norm: np.ndarray,
    meta: List[Dict[str, Any]],
    train_cfg: TrainConfig,
    arch_cfg: AeArchConfig,
) -> Dict[str, Any]:
    X_train, X_val, X_test_holdout, _split_info = make_global_date_chronological_split(
        X_norm,
        meta,
        test_frac=0.10,
        val_frac_within_train=0.10,
    )

    ds_train = XDataset(X_train)
    ds_val = XDataset(X_val)

    dl_train = torch.utils.data.DataLoader(
        ds_train, batch_size=train_cfg.batch_size, shuffle=True,
        num_workers=train_cfg.num_workers, pin_memory=train_cfg.device.startswith("cuda")
    )
    dl_val = torch.utils.data.DataLoader(
        ds_val, batch_size=train_cfg.batch_size, shuffle=False,
        num_workers=train_cfg.num_workers, pin_memory=train_cfg.device.startswith("cuda")
    )

    model = TeCnnAae(input_len=TARGET_LEN, arch=arch_cfg).to(train_cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
    mse = nn.MSELoss()

    best_val_total = float("inf")
    best_val_recon = None
    best_val_adv = None
    best_state = None

    for ep in range(train_cfg.epochs):
        model.train()
        for xb in dl_train:
            xb = xb.to(train_cfg.device).float().unsqueeze(1)
            ytrend = ytrend_from_intraday_batch(xb, angle_clip_deg=train_cfg.angle_clip_deg)

            opt.zero_grad(set_to_none=True)
            x_hat, trend_hat, _ = model(xb)

            L_recon = mse(x_hat, xb)
            L_adv = mse(trend_hat, ytrend)
            L_total = train_cfg.lambda_recon * L_recon + train_cfg.lambda_adv * L_adv

            L_total.backward()
            opt.step()

        model.eval()
        val_total_list, val_recon_list, val_adv_list = [], [], []
        with torch.no_grad():
            for xb in dl_val:
                xb = xb.to(train_cfg.device).float().unsqueeze(1)
                ytrend = ytrend_from_intraday_batch(xb, angle_clip_deg=train_cfg.angle_clip_deg)

                x_hat, trend_hat, _ = model(xb)

                L_recon = mse(x_hat, xb)
                L_adv = mse(trend_hat, ytrend)
                L_total = train_cfg.lambda_recon * L_recon + train_cfg.lambda_adv * L_adv

                val_total_list.append(float(L_total.item()))
                val_recon_list.append(float(L_recon.item()))
                val_adv_list.append(float(L_adv.item()))

        mean_total = float(np.mean(val_total_list))
        mean_recon = float(np.mean(val_recon_list))
        mean_adv = float(np.mean(val_adv_list))

        if mean_total < best_val_total:
            best_val_total = mean_total
            best_val_recon = mean_recon
            best_val_adv = mean_adv
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return {
        "best_val_total": best_val_total,
        "best_val_recon": best_val_recon,
        "best_val_adv": best_val_adv,
        "state_dict": best_state,
    }
def train_tecnn_aae(
    X_norm: np.ndarray,
    meta: List[Dict[str, Any]],
    cfg: TrainConfig,
    model_out: Path,
    arch_cfg: Optional[AeArchConfig] = None,
) -> TeCnnAae:
    """
    Treina o TE-CNN-AAE.

    Se arch_cfg não for informado, usa a arquitetura final da Tabela IIa.
    """
    set_seed(cfg.seed)
    ensure_dir(model_out.parent)

    X_train, X_val, X_test_holdout, split_info = make_global_date_chronological_split(
        X_norm,
        meta,
        test_frac=0.10,
        val_frac_within_train=0.10,
    )

    ds_train = XDataset(X_train)
    ds_val = XDataset(X_val)

    loader_train = torch.utils.data.DataLoader(
        ds_train,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.device.startswith("cuda"),
    )
    loader_val = torch.utils.data.DataLoader(
        ds_val,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
    )

    if arch_cfg is None:
        arch_cfg = AeArchConfig(
            filters=(16, 32, 128),
            kernels=(7, 7, 7),
            z_dim=cfg.z_dim,
            dropout=cfg.dropout,
        )

    model = TeCnnAae(
        input_len=TARGET_LEN,
        arch=arch_cfg,
    ).to(cfg.device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )
    mse = nn.MSELoss()

    for ep in range(1, cfg.epochs + 1):
        model.train()
        loss_sum = 0.0
        recon_sum = 0.0
        adv_sum = 0.0
        n_batches = 0

        for xb in loader_train:
            xb = xb.to(cfg.device).float().unsqueeze(1)
            ytrend = ytrend_from_intraday_batch(
                xb,
                angle_clip_deg=cfg.angle_clip_deg
            )

            opt.zero_grad(set_to_none=True)
            x_hat, trend_hat, _ = model(xb)

            L_recon = mse(x_hat, xb)
            L_adv = mse(trend_hat, ytrend)
            L_total = cfg.lambda_recon * L_recon + cfg.lambda_adv * L_adv

            L_total.backward()
            opt.step()

            loss_sum += float(L_total.item())
            recon_sum += float(L_recon.item())
            adv_sum += float(L_adv.item())
            n_batches += 1

        model.eval()
        val_loss_sum = 0.0
        val_batches = 0
        with torch.no_grad():
            for xb in loader_val:
                xb = xb.to(cfg.device).float().unsqueeze(1)
                ytrend = ytrend_from_intraday_batch(xb, angle_clip_deg=cfg.angle_clip_deg)
                x_hat, trend_hat, _ = model(xb)
                L_recon = mse(x_hat, xb)
                L_adv = mse(trend_hat, ytrend)
                val_loss_sum += float((cfg.lambda_recon * L_recon + cfg.lambda_adv * L_adv).item())
                val_batches += 1

        print(
            f"[epoch {ep:03d}/{cfg.epochs}] "
            f"loss={loss_sum/max(1,n_batches):.6f} "
            f"recon={recon_sum/max(1,n_batches):.6f} "
            f"adv={adv_sum/max(1,n_batches):.6f} "
            f"val_loss={val_loss_sum/max(1,val_batches):.6f}"
        )

    cfg_to_save = asdict(cfg)
    cfg_to_save["te_arch"] = model.arch.as_dict()
    cfg_to_save["arch_version"] = "tecnn_v2_tconv_decoder"
    cfg_to_save["split_protocol"] = split_info

    torch.save(
        {
            "arch_cfg": model.arch.as_dict(),
            "state_dict": model.state_dict(),
            "config": cfg_to_save,
            "model_type": "te_cnn_aae",
        },
        model_out
    )
    print(f"[OK] modelo salvo em {model_out}")

    return model

# ============================================================================
# TREINO DO CNN-AE (ablação)
# ============================================================================
#
# Aqui a diferença para o TE-CNN-AAE é simples:
# - não calculamos y_trend
# - não usamos trend_hat
# - a loss total vira apenas:
#
#       L_total = L_recon = MSE(x_hat, x)
# ============================================================================
def fit_cnnae_trial(
    X_norm: np.ndarray,
    meta: List[Dict[str, Any]],
    train_cfg: TrainConfig,
    arch_cfg: AeArchConfig,
) -> Dict[str, Any]:
    X_train, X_val, X_test_holdout, _split_info = make_global_date_chronological_split(
        X_norm,
        meta,
        test_frac=0.10,
        val_frac_within_train=0.10,
    )

    ds_train = XDataset(X_train)
    ds_val = XDataset(X_val)

    dl_train = torch.utils.data.DataLoader(
        ds_train, batch_size=train_cfg.batch_size, shuffle=True,
        num_workers=train_cfg.num_workers, pin_memory=train_cfg.device.startswith("cuda")
    )
    dl_val = torch.utils.data.DataLoader(
        ds_val, batch_size=train_cfg.batch_size, shuffle=False,
        num_workers=train_cfg.num_workers, pin_memory=train_cfg.device.startswith("cuda")
    )

    model = CnnAe(input_len=TARGET_LEN, arch=arch_cfg).to(train_cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
    mse = nn.MSELoss()

    best_val_recon = float("inf")
    best_state = None

    for ep in range(train_cfg.epochs):
        model.train()
        for xb in dl_train:
            xb = xb.to(train_cfg.device).float().unsqueeze(1)

            opt.zero_grad(set_to_none=True)
            x_hat, _ = model(xb)
            L_recon = mse(x_hat, xb)
            L_recon.backward()
            opt.step()

        model.eval()
        val_recon_list = []
        with torch.no_grad():
            for xb in dl_val:
                xb = xb.to(train_cfg.device).float().unsqueeze(1)
                x_hat, _ = model(xb)
                L_recon = mse(x_hat, xb)
                val_recon_list.append(float(L_recon.item()))

        mean_recon = float(np.mean(val_recon_list))
        if mean_recon < best_val_recon:
            best_val_recon = mean_recon
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return {
        "best_val_recon": best_val_recon,
        "state_dict": best_state,
    }

def train_cnn_ae(
    X_norm: np.ndarray,
    meta: List[Dict[str, Any]],
    cfg: TrainConfig,
    model_out: Path,
    arch_cfg: Optional[AeArchConfig] = None,
) -> CnnAe:
    """
    Treina o CNN-AE (ablação sem Trend Discriminator).

    Se arch_cfg não for informado, usa a arquitetura final da Tabela IIb.
    """
    set_seed(cfg.seed)
    ensure_dir(model_out.parent)

    X_train_ae, X_val_ae, X_test_holdout, split_info = make_global_date_chronological_split(
        X_norm,
        meta,
        test_frac=0.10,
        val_frac_within_train=0.10,
    )

    ds_train = XDataset(X_train_ae)
    ds_val = XDataset(X_val_ae)

    loader_train = torch.utils.data.DataLoader(
        ds_train,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.device.startswith("cuda"),
    )
    loader_val = torch.utils.data.DataLoader(
        ds_val,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
    )

    if arch_cfg is None:
        arch_cfg = AeArchConfig(
            filters=(16, 64, 128),
            kernels=(7, 7, 25),
            z_dim=cfg.z_dim,
            dropout=cfg.dropout,
        )

    model = CnnAe(
        input_len=TARGET_LEN,
        arch=arch_cfg,
    ).to(cfg.device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )
    mse = nn.MSELoss()

    for ep in range(1, cfg.epochs + 1):
        model.train()
        loss_sum = 0.0
        n_batches = 0

        for xb in loader_train:
            xb = xb.to(cfg.device).float().unsqueeze(1)
            opt.zero_grad(set_to_none=True)
            x_hat, _ = model(xb)
            L_recon = mse(x_hat, xb)
            L_recon.backward()
            opt.step()

            loss_sum += float(L_recon.item())
            n_batches += 1

        model.eval()
        val_loss_sum = 0.0
        val_batches = 0
        with torch.no_grad():
            for xb in loader_val:
                xb = xb.to(cfg.device).float().unsqueeze(1)
                x_hat, _ = model(xb)
                L_recon = mse(x_hat, xb)
                val_loss_sum += float(L_recon.item())
                val_batches += 1

        print(
            f"[CNN-AE epoch {ep:03d}/{cfg.epochs}] "
            f"recon={loss_sum/max(1,n_batches):.6f} "
            f"val_recon={val_loss_sum/max(1,val_batches):.6f}"
        )

    cfg_to_save = asdict(cfg)
    cfg_to_save["cnn_ae_arch"] = model.arch.as_dict()
    cfg_to_save["arch_version"] = "cnn_ae_table2b_v2"
    cfg_to_save["split_protocol"] = split_info

    torch.save(
        {
            "arch_cfg": model.arch.as_dict(),
            "state_dict": model.state_dict(),
            "config": cfg_to_save,
            "model_type": "cnn_ae",
        },
        model_out
    )
    print(f"[OK] modelo CNN-AE salvo em {model_out}")

    return model
# ============================================================================
# EXTRAÇÃO DE EMBEDDINGS
# ============================================================================
# --------------------------------------------------------------------------
# RESUMO DA EXTRAÇÃO DE EMBEDDINGS
# --------------------------------------------------------------------------
# Aqui NÃO usamos decoder nem Trend Discriminator.
# Usamos apenas:
#   X_norm -> encoder -> z
#
# Saída:
#   z [N,z_dim]
#
# Essa é a representação que será exportada para CSV/ARFF.
# --------------------------------------------------------------------------

@torch.no_grad()
def extract_embeddings(model: TeCnnAae, X_norm: np.ndarray, device: str, batch_size: int = 256) -> np.ndarray:
    """
    Extrai os embeddings z usando apenas o encoder.

    FLUXO
    -----
    Passo 1: carregar X_norm em batches
    Passo 2: aplicar encoder
    Passo 3: acumular z de todos os batches
    Passo 4: concatenar tudo em uma única matriz [N,z_dim]

    Essa saída é a mais importante para a futura integração com SMOClust.
    """
    ds = XDataset(X_norm)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False)

    model.eval().to(device)
    zs = []

    for xb in loader:
        # Passo 1: preparar batch
        xb = xb.to(device).float().unsqueeze(1)

        # Passo 2: extrair z
        z = model.encode(xb)

        # Passo 3: guardar em memória CPU
        zs.append(z.cpu().numpy())

    # Passo 4: concatenar embeddings
    return np.concatenate(zs, axis=0)


# ============================================================================
# LABELS 3-CLASSES POR ENTROPIA MÁXIMA
# ============================================================================

def entropy_from_counts(counts: np.ndarray) -> float:
    """
    Calcula a entropia de Shannon a partir de contagens.

    Exemplo:
        counts = [n_inc, n_dec, n_no]

    Quanto mais balanceadas as classes, maior tende a ser a entropia.

    Parameters
    ----------
    counts : np.ndarray
        Vetor de contagens não negativas.

    Returns
    -------
    float
        Valor da entropia.
    """
    n = float(counts.sum())
    if n <= 0:
        return 0.0
    p = counts.astype(np.float64) / n
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def max_tau_by_histogram(delta: np.ndarray, bins: int = 10, coverage: float = 0.85) -> float:
    """
    Calcula um tau máximo usando histograma sobre |delta|.

    Ideia:
    - constrói histograma das magnitudes |delta|
    - ordena bins por frequência
    - acumula até cobrir 85% dos exemplos
    - usa a borda superior do bin escolhido como tau máximo

    Isso replica a lógica descrita no paper para limitar a busca do threshold.

    Parameters
    ----------
    delta : np.ndarray
        Vetor de variações diárias de fechamento.

    bins : int
        Número de bins do histograma.

    coverage : float
        Cobertura acumulada desejada.

    Returns
    -------
    float
        Limite superior da busca de tau.
    """
    x = np.abs(delta).astype(np.float64)
    if x.size == 0:
        return 0.0

    mn, mx = float(x.min()), float(x.max())
    if mx <= mn + 1e-12:
        return mx

    counts, edges = np.histogram(x, bins=bins)
    order = np.argsort(counts)[::-1]
    cum = np.cumsum(counts[order])
    target = coverage * x.size
    j = int(np.searchsorted(cum, target, side="left"))
    j = min(j, len(order) - 1)
    chosen = order[j]
    return float(edges[chosen + 1])


def find_tau_max_entropy(delta: np.ndarray, step: float = 0.01) -> Tuple[float, float]:
    """
    Procura o tau que maximiza a entropia das 3 classes:
    - Increase
    - Decrease
    - NoAction

    Regra:
        inc : delta >  tau
        dec : delta < -tau
        no  : |delta| <= tau

    Returns
    -------
    tau_best : float
        Melhor threshold encontrado.

    H_best : float
        Entropia correspondente.
    """
    max_tau = max_tau_by_histogram(delta, bins=10, coverage=0.85)
    best_tau, best_H = 0.0, -1.0

    tau = 0.0
    while tau <= max_tau + 1e-12:
        n_inc = int((delta > tau).sum())
        n_dec = int((delta < -tau).sum())
        n_no = int((np.abs(delta) <= tau).sum())
        H = entropy_from_counts(np.array([n_inc, n_dec, n_no], dtype=np.float64))
        if H > best_H:
            best_H, best_tau = H, float(tau)
        tau += step

    return best_tau, best_H


def label_3class_from_tau(delta: np.ndarray, tau: float) -> np.ndarray:
    """
    Gera labels 3-classes com base em tau.

    Mapeamento:
    - 0 : Decrease
    - 1 : NoAction
    - 2 : Increase
    """
    y = np.ones(delta.shape[0], dtype=np.int64)
    y[delta > tau] = 2
    y[delta < -tau] = 0
    return y


# ============================================================================
# EXPORTAÇÃO
# ============================================================================

def write_arff_embeddings(
    path: Path,
    Z: np.ndarray,
    y: Optional[np.ndarray] = None,
    class_names: Optional[Tuple[str, ...]] = None,
    relation: str = "te_cnn_aae_embeddings",
) -> None:
    """
    Exporta embeddings para ARFF.

    Esse formato é útil para integração futura com MOA/SMOClust.

    Parameters
    ----------
    path : Path
        Caminho do ARFF de saída.

    Z : np.ndarray
        Matriz [N,D] de embeddings.

    y : Optional[np.ndarray]
        Labels opcionais.

    class_names : Optional[Tuple[str, ...]]
        Nomes textuais das classes.

    relation : str
        Nome da relação ARFF.
    """
    if y is not None and class_names is None:
        class_names = ("Decrease", "NoAction", "Increase")

    with open(path, "w", encoding="utf-8") as f:
        f.write(f"@RELATION {relation}\n\n")
        for j in range(Z.shape[1]):
            f.write(f"@ATTRIBUTE z{j} NUMERIC\n")
        if y is not None:
            f.write(f"@ATTRIBUTE class {{{','.join(class_names)}}}\n")
        f.write("\n@DATA\n")
        for i in range(Z.shape[0]):
            row = ",".join(f"{Z[i, j]:.6f}" for j in range(Z.shape[1]))
            if y is not None:
                row += "," + class_names[int(y[i])]
            f.write(row + "\n")

# ============================================================================
# LSTM DOWNSTREAM (classificação 3-classes)
# ============================================================================
#
# O objetivo desta seção é aproximar a parte downstream do artigo:
# - representação PRD  : usa X_norm (79 features por dia)
# - representação TE   : usa embeddings z extraídos pelo TE-CNN-AAE
# - baseline aleatório : Random Classifier
#
# A lógica geral é:
#
#   1) carregar a representação diária de um ticker
#   2) gerar labels 3-classes a partir do fechamento bruto diário
#      usando tau calculado APENAS no treino
#   3) transformar o problema em janelas de W dias
#   4) treinar um LSTM de 1 camada
#   5) fazer grid search
#   6) rodar 100 execuções com os melhores hiperparâmetros
#   7) salvar resultados em JSON
#
# Isso reproduz a metodologia do artigo de forma aproximada para:
# - PRD
# - TE-CNN-AAE
# - RC
#
# Ainda NÃO reproduz:
# - CNN-AE (ablação sem discriminator), porque esse modelo ainda não existe
#   no seu código atual.
# ============================================================================




try:
    from scipy.stats import mannwhitneyu
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


# --------------------------------------------------------------------------
# CONFIGURAÇÕES DO GRID SEARCH DO LSTM
# --------------------------------------------------------------------------
def iter_article_ae_grid():
    for f1 in (16, 32):
        for f2 in (32, 64):
            for f3 in (64, 128):
                for k1 in (7, 13, 25):
                    for k2 in (7, 13, 25):
                        for k3 in (7, 13, 25):
                            yield AeArchConfig(
                                filters=(f1, f2, f3),
                                kernels=(k1, k2, k3),
                                z_dim=64,
                                dropout=0.1,
                            )
def ae_trial_id(model_kind: str, arch: AeArchConfig) -> str:
    f1, f2, f3 = arch.filters
    k1, k2, k3 = arch.kernels
    return f"{model_kind}_f{f1}-{f2}-{f3}_k{k1}-{k2}-{k3}"


def save_json(obj: Dict[str, Any], path: Path):
    ensure_dir(path.parent)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
def search_autoencoder_grid(
    model_kind: str,
    X_norm: np.ndarray,
    meta: List[Dict[str, Any]],
    out_dir: Path,
    train_cfg: TrainConfig,
) -> Dict[str, Any]:
    """
    model_kind:
      'te' ou 'ae'
    """
    ensure_dir(out_dir)

    results_dir = out_dir / f"{model_kind}_trials"
    best_ckpt_dir = out_dir / f"{model_kind}_best"
    ensure_dir(results_dir)
    ensure_dir(best_ckpt_dir)

    best_score = float("inf")
    best_arch = None
    best_metrics = None
    best_state = None

    for arch in iter_article_ae_grid():
        tid = ae_trial_id(model_kind, arch)
        trial_json = results_dir / f"{tid}.json"

        # resume: se já existe, reutiliza
        if trial_json.exists():

            with open(trial_json, "r", encoding="utf-8") as f:
                metrics = json.load(f)
        else:
            if model_kind == "te":
                fit = fit_tecnn_trial(X_norm, meta, train_cfg, arch)
                metrics = {
                    "trial_id": tid,
                    "model_kind": model_kind,
                    "arch": arch.as_dict(),
                    "score": fit["best_val_total"],
                    "best_val_total": fit["best_val_total"],
                    "best_val_recon": fit["best_val_recon"],
                    "best_val_adv": fit["best_val_adv"],
                }
                state_dict = fit["state_dict"]
            elif model_kind == "ae":
                fit = fit_cnnae_trial(X_norm, meta, train_cfg, arch)
                metrics = {
                    "trial_id": tid,
                    "model_kind": model_kind,
                    "arch": arch.as_dict(),
                    "score": fit["best_val_recon"],
                    "best_val_recon": fit["best_val_recon"],
                }
                state_dict = fit["state_dict"]
            else:
                raise ValueError(f"model_kind inválido: {model_kind}")

            save_json(metrics, trial_json)

            # salva checkpoint do melhor trial até agora
            if metrics["score"] < best_score:
                best_score = metrics["score"]
                best_arch = arch
                best_metrics = metrics
                best_state = state_dict

                torch.save(
                    {
                        "state_dict": best_state,
                        "config": asdict(train_cfg),
                        "model_type": "te_cnn_aae" if model_kind == "te" else "cnn_ae",
                        "arch_cfg": best_arch.as_dict(),
                        "search_score": best_score,
                    },
                    best_ckpt_dir / f"best_{model_kind}.pt"
                )

                save_json(
                    {
                        "best_score": best_score,
                        "best_arch": best_arch.as_dict(),
                        "best_metrics": best_metrics,
                    },
                    out_dir / f"best_{model_kind}.json"
                )

        # atualiza melhor mesmo ao restaurar
        if metrics["score"] < best_score:
            best_score = metrics["score"]
            best_arch = AeArchConfig(
                filters=tuple(metrics["arch"]["filters"]),
                kernels=tuple(metrics["arch"]["kernels"]),
                z_dim=int(metrics["arch"]["z_dim"]),
                dropout=float(metrics["arch"]["dropout"]),
            )
            best_metrics = metrics

    return {
        "best_score": best_score,
        "best_arch": best_arch.as_dict() if best_arch else None,
        "best_metrics": best_metrics,
    }

@dataclass
class LstmGrid:
    batch_sizes: Tuple[int, ...] = (4, 32, 64)
    windows: Tuple[int, ...] = (5, 15, 30)
    lstm_units: Tuple[int, ...] = (16, 64, 128)
    learning_rates: Tuple[float, ...] = (1e-2, 1e-3, 1e-4)


@dataclass
class LstmRunConfig:
    """
    Configuração fixa de um treino do LSTM.
    """
    window: int
    batch_size: int
    lstm_units: int
    learning_rate: float
    epochs: int = 20
    patience: int = 5
    seed: int = 123
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# --------------------------------------------------------------------------
# MODELO LSTM
# --------------------------------------------------------------------------
class SingleLSTMClassifier(nn.Module):
    """
    Classificador LSTM com uma única camada + camada densa final.

    Entrada:
        X -> [B, W, D]
            B = batch size
            W = tamanho da janela (número de dias passados)
            D = dimensão da representação diária
                - 79 para PRD
                - z_dim para embeddings TE

    Saída:
        logits -> [B, 3]
            classes = {Decrease, NoAction, Increase}
    """
    def __init__(self, input_dim: int, hidden_units: int, num_classes: int = 3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_units,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_units, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, W, D]

        # 1) roda a sequência na LSTM
        out, (h_n, c_n) = self.lstm(x)

        # 2) pega o último hidden state da última camada
        #    h_n shape: [num_layers, B, hidden_units]
        h_last = h_n[-1]  # [B, hidden_units]

        # 3) projeta para 3 classes
        logits = self.fc(h_last)
        return logits


# --------------------------------------------------------------------------
# HELPERS DE CARGA DE REPRESENTAÇÕES
# --------------------------------------------------------------------------
def load_prd_representation_for_ticker(data_dir: Path, ticker: str) -> pd.DataFrame:
    """
    Carrega a representação PRD de um ticker.

    PRD, no contexto do seu pipeline atual, será:
        X_norm [N,79]

    Retorno:
        DataFrame com colunas:
            date, f0, f1, ..., f78
    """
    fp = data_dir / ticker / "data_79.npz"
    if not fp.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {fp}")

    d = np.load(fp, allow_pickle=True)
    X = d["X_norm"].astype(np.float32)
    dates = d["dates"].astype(str)

    df = pd.DataFrame({"date": dates})
    for j in range(X.shape[1]):
        df[f"f{j}"] = X[:, j]
    return df.sort_values("date").reset_index(drop=True)


def load_te_representation_for_ticker(emb_csv: Path, ticker: str) -> pd.DataFrame:
    """
    Carrega a representação TE-CNN-AAE de um ticker a partir do CSV de embeddings.

    Espera um CSV gerado por cmd_embed com colunas:
        ticker, date, eod_close_raw, z0, z1, ..., z{D-1}

    Retorno:
        DataFrame com colunas:
            date, f0, f1, ..., f{D-1}
    """
    df = pd.read_csv(emb_csv)
    df = df[df["ticker"] == ticker].copy()
    if df.empty:
        raise RuntimeError(f"Nenhum embedding encontrado para o ticker {ticker} em {emb_csv}")

    z_cols = [c for c in df.columns if c.startswith("z")]
    out = pd.DataFrame({"date": df["date"].astype(str)})
    for i, c in enumerate(z_cols):
        out[f"f{i}"] = df[c].astype(np.float32)
    return out.sort_values("date").reset_index(drop=True)


# --------------------------------------------------------------------------
# LABELS 3-CLASSES COM TAU CALCULADO SÓ NO TREINO
# --------------------------------------------------------------------------
def build_labels_for_ticker_train_test(
    data_dir: Path,
    ticker: str,
    train_end: str = "2022-12-31",
    test_start: str = "2023-01-01",
    test_end: str = "2023-01-31",
) -> Tuple[pd.DataFrame, float, float]:
    """
    Gera os labels 3-classes de um ticker, calculando tau SOMENTE no treino.

    Fluxo:
    1) carrega dates + eod_close_raw
    2) calcula delta diário
    3) usa apenas deltas com date <= train_end para encontrar tau ótimo
    4) aplica esse tau em todos os deltas
    5) devolve um DataFrame com:
         date, delta, label, split

    IMPORTANTE:
    - o label da data t representa o movimento do fechamento do dia t
      em relação ao fechamento do dia t-1
    - portanto, os labels começam na segunda data disponível
    """
    dates, eod = load_per_ticker_close_series(data_dir, ticker)

    if len(eod) < 2:
        raise RuntimeError(f"Ticker {ticker} não tem dados suficientes para gerar labels.")

    # 1) delta diário
    delta = np.diff(eod)

    # 2) as datas associadas aos labels começam da 2ª data
    label_dates = np.asarray(dates[1:], dtype=str)

    # 3) separa deltas de treino para calcular tau
    train_mask = label_dates <= train_end
    delta_train = delta[train_mask]

    if delta_train.size == 0:
        raise RuntimeError(f"Ticker {ticker} não possui dados no período de treino para calcular tau.")

    tau, H = find_tau_max_entropy(delta_train, step=0.01)

    # 4) aplica tau em todos os deltas
    y = label_3class_from_tau(delta, tau)

    # 5) define split por data do label
    split = np.full(label_dates.shape[0], "discard", dtype=object)
    split[label_dates <= train_end] = "train"
    split[(label_dates >= test_start) & (label_dates <= test_end)] = "test"

    df = pd.DataFrame({
        "date": label_dates,
        "delta": delta.astype(np.float64),
        "label": y.astype(np.int64),
        "split": split,
    })

    return df, float(tau), float(H)


# --------------------------------------------------------------------------
# CONSTRUÇÃO DE JANELAS SUPERVISIONADAS
# --------------------------------------------------------------------------
def make_windowed_dataset(
    rep_df: pd.DataFrame,
    labels_df: pd.DataFrame,
    window: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Constrói um dataset supervisionado para o LSTM.

    Lógica:
    - rep_df tem uma representação por dia (PRD ou TE)
    - labels_df tem um label por dia de destino
    - para prever o label do dia t, usamos os W dias anteriores:
          [t-W, ..., t-1]

    Exemplo:
      window = 5
      alvo = label do dia 2023-01-10
      entrada = representações dos dias:
               2023-01-03, 2023-01-04, 2023-01-05, 2023-01-06, 2023-01-09

    Retorno:
      X -> [N, W, D]
      y -> [N]
      dates_target -> [N]
    """
    rep_df = rep_df.sort_values("date").reset_index(drop=True)
    labels_df = labels_df.sort_values("date").reset_index(drop=True)

    label_map = {
        row["date"]: int(row["label"])
        for _, row in labels_df.iterrows()
        if row["split"] in ("train", "test")
    }

    all_dates = rep_df["date"].astype(str).tolist()
    feat_cols = [c for c in rep_df.columns if c.startswith("f")]
    feats = rep_df[feat_cols].to_numpy(dtype=np.float32)

    X_list, y_list, d_list = [], [], []

    # target index começa em "window" porque precisamos de W dias passados
    for idx in range(window, len(all_dates)):
        target_date = all_dates[idx]

        # só constrói exemplo se houver label para essa data
        if target_date not in label_map:
            continue

        x_win = feats[idx - window: idx]  # W dias anteriores
        if x_win.shape[0] != window:
            continue

        y = label_map[target_date]

        X_list.append(x_win)
        y_list.append(y)
        d_list.append(target_date)

    if not X_list:
        return (
            np.empty((0, window, len(feat_cols)), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
            np.empty((0,), dtype=str),
        )

    return (
        np.stack(X_list).astype(np.float32),
        np.asarray(y_list, dtype=np.int64),
        np.asarray(d_list, dtype=str),
    )


def split_train_val_test_by_date(
    X: np.ndarray,
    y: np.ndarray,
    target_dates: np.ndarray,
    train_end: str,
    test_start: str,
    test_end: str,
    val_frac_inside_train: float = 0.10,
):
    """
    Divide o dataset em:
      - treino
      - validação
      - teste

    Estratégia:
    - primeiro separamos por datas de target:
        treino: date <= train_end
        teste : test_start <= date <= test_end
    - dentro do conjunto de treino, reservamos os 10% finais para validação,
      preservando a ordem temporal
    """
    train_mask = target_dates <= train_end
    test_mask = (target_dates >= test_start) & (target_dates <= test_end)

    X_train_full, y_train_full = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    if len(X_train_full) == 0 or len(X_test) == 0:
        return None

    n_train_full = len(X_train_full)
    n_val = max(1, int(round(val_frac_inside_train * n_train_full)))

    if n_train_full <= n_val:
        return None

    split_idx = n_train_full - n_val

    X_train = X_train_full[:split_idx]
    y_train = y_train_full[:split_idx]

    X_val = X_train_full[split_idx:]
    y_val = y_train_full[split_idx:]

    return X_train, y_train, X_val, y_val, X_test, y_test


# --------------------------------------------------------------------------
# TREINO DE UMA EXECUÇÃO DO LSTM
# --------------------------------------------------------------------------
class WindowDataset(torch.utils.data.Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)


@torch.no_grad()
def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    pred = logits.argmax(dim=1)
    return float((pred == y).float().mean().item())


def train_one_lstm_run(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    run_cfg: LstmRunConfig,
) -> Tuple[float, float]:
    """
    Treina UMA execução do LSTM.

    Retorna:
      best_val_acc
      test_acc usando o melhor checkpoint de validação
    """
    set_seed(run_cfg.seed)

    device = run_cfg.device
    input_dim = X_train.shape[2]

    ds_train = WindowDataset(X_train, y_train)
    ds_val = WindowDataset(X_val, y_val)
    ds_test = WindowDataset(X_test, y_test)

    dl_train = torch.utils.data.DataLoader(ds_train, batch_size=run_cfg.batch_size, shuffle=True)
    dl_val = torch.utils.data.DataLoader(ds_val, batch_size=run_cfg.batch_size, shuffle=False)
    dl_test = torch.utils.data.DataLoader(ds_test, batch_size=run_cfg.batch_size, shuffle=False)

    model = SingleLSTMClassifier(
        input_dim=input_dim,
        hidden_units=run_cfg.lstm_units,
        num_classes=3,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=run_cfg.learning_rate)
    criterion = nn.CrossEntropyLoss()

    best_val = -1.0
    best_state = None
    patience_left = run_cfg.patience

    # ------------------------------------------------------------------
    # loop de épocas
    # ------------------------------------------------------------------
    for ep in range(run_cfg.epochs):
        model.train()

        for xb, yb in dl_train:
            xb = xb.to(device).float()
            yb = yb.to(device).long()

            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()

        # ------------------------
        # validação
        # ------------------------
        model.eval()
        val_accs = []

        with torch.no_grad():
            for xb, yb in dl_val:
                xb = xb.to(device).float()
                yb = yb.to(device).long()
                logits = model(xb)
                val_accs.append(accuracy_from_logits(logits, yb))

        mean_val = float(np.mean(val_accs)) if val_accs else 0.0

        if mean_val > best_val:
            best_val = mean_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = run_cfg.patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    # restaura melhor checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)

    # ------------------------
    # teste
    # ------------------------
    model.eval()
    test_accs = []

    with torch.no_grad():
        for xb, yb in dl_test:
            xb = xb.to(device).float()
            yb = yb.to(device).long()
            logits = model(xb)
            test_accs.append(accuracy_from_logits(logits, yb))

    mean_test = float(np.mean(test_accs)) if test_accs else 0.0
    return float(best_val), float(mean_test)


# --------------------------------------------------------------------------
# GRID SEARCH
# --------------------------------------------------------------------------
def grid_search_lstm_for_representation(
    rep_df: pd.DataFrame,
    labels_df: pd.DataFrame,
    grid: LstmGrid,
    train_end: str,
    test_start: str,
    test_end: str,
    n_repeats_per_grid_point: int = 10,
    epochs: int = 20,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> Dict[str, Any]:
    """
    Faz o grid search do LSTM para UMA representação e UM ticker.

    Critério:
    - para cada combinação de hiperparâmetros:
        - executa 10 vezes
        - mede acurácia média de validação
    - escolhe a combinação com melhor média de validação

    Retorno:
      dicionário com:
        best_params
        best_val_acc
    """
    best = {
        "best_val_acc": -1.0,
        "best_params": None,
    }

    for window, batch_size, lstm_units, lr in product(
        grid.windows, grid.batch_sizes, grid.lstm_units, grid.learning_rates
    ):
        # 1) constrói as janelas para esse window
        X, y, dts = make_windowed_dataset(rep_df, labels_df, window=window)

        split = split_train_val_test_by_date(
            X, y, dts,
            train_end=train_end,
            test_start=test_start,
            test_end=test_end,
            val_frac_inside_train=0.10,
        )

        if split is None:
            continue

        X_train, y_train, X_val, y_val, X_test, y_test = split

        # 2) repete 10 vezes
        val_scores = []

        for rep in range(n_repeats_per_grid_point):
            run_cfg = LstmRunConfig(
                window=window,
                batch_size=batch_size,
                lstm_units=lstm_units,
                learning_rate=lr,
                epochs=epochs,
                patience=5,
                seed=123 + rep,
                device=device,
            )

            val_acc, _ = train_one_lstm_run(
                X_train, y_train,
                X_val, y_val,
                X_test, y_test,
                run_cfg,
            )
            val_scores.append(val_acc)

        mean_val = float(np.mean(val_scores)) if val_scores else -1.0

        if mean_val > best["best_val_acc"]:
            best["best_val_acc"] = mean_val
            best["best_params"] = {
                "batch_size": batch_size,
                "window": window,
                "learning_rate": lr,
                "lstm_units": lstm_units,
            }

    return best


# --------------------------------------------------------------------------
# 100 EXECUÇÕES FINAIS
# --------------------------------------------------------------------------
def evaluate_representation_100x(
    rep_df: pd.DataFrame,
    labels_df: pd.DataFrame,
    best_params: Dict[str, Any],
    train_end: str,
    test_start: str,
    test_end: str,
    n_runs: int = 100,
    epochs: int = 20,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> List[float]:
    """
    Avalia uma representação usando os melhores hiperparâmetros em 100 execuções.
    """
    window = int(best_params["window"])

    X, y, dts = make_windowed_dataset(rep_df, labels_df, window=window)
    split = split_train_val_test_by_date(
        X, y, dts,
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
        val_frac_inside_train=0.10,
    )
    if split is None:
        return []

    X_train, y_train, X_val, y_val, X_test, y_test = split

    accs = []
    for run in range(n_runs):
        run_cfg = LstmRunConfig(
            window=window,
            batch_size=int(best_params["batch_size"]),
            lstm_units=int(best_params["lstm_units"]),
            learning_rate=float(best_params["learning_rate"]),
            epochs=epochs,
            patience=5,
            seed=1000 + run,
            device=device,
        )

        _, test_acc = train_one_lstm_run(
            X_train, y_train,
            X_val, y_val,
            X_test, y_test,
            run_cfg,
        )
        accs.append(float(test_acc))

    return accs


# --------------------------------------------------------------------------
# RANDOM BASELINE
# --------------------------------------------------------------------------
def evaluate_random_classifier_100x(
    labels_df: pd.DataFrame,
    rep_df: pd.DataFrame,
    window: int,
    train_end: str,
    test_start: str,
    test_end: str,
    n_runs: int = 100,
) -> List[float]:
    """
    Baseline aleatório:
    - usa a MESMA construção de janelas apenas para obter y_test
    - depois faz previsões aleatórias uniformes em {0,1,2}
    """
    X, y, dts = make_windowed_dataset(rep_df, labels_df, window=window)
    split = split_train_val_test_by_date(
        X, y, dts,
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
        val_frac_inside_train=0.10,
    )
    if split is None:
        return []

    _, _, _, _, _, y_test = split

    rng = np.random.default_rng(12345)
    accs = []
    for _ in range(n_runs):
        pred = rng.integers(low=0, high=3, size=len(y_test))
        acc = float((pred == y_test).mean()) if len(y_test) > 0 else 0.0
        accs.append(acc)
    return accs


# --------------------------------------------------------------------------
# MANN-WHITNEY (opcional)
# --------------------------------------------------------------------------
def mannwhitney_one_tailed_greater(x: List[float], y: List[float]) -> Optional[float]:
    """
    Teste unilateral H1: x > y
    Retorna p-value ou None se scipy não estiver disponível.
    """
    if not SCIPY_AVAILABLE:
        return None
    res = mannwhitneyu(x, y, alternative="greater")
    return float(res.pvalue)


# --------------------------------------------------------------------------
# EXPERIMENTO COMPLETO POR TICKER
# --------------------------------------------------------------------------
def run_lstm_downstream_for_ticker(
    data_dir: Path,
    emb_csv: Path,
    out_dir: Path,
    ticker: str,
    ae_emb_csv: Optional[Path] = None,
    train_end: str = "2022-12-31",
    test_start: str = "2023-01-01",
    test_end: str = "2023-01-31",
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    epochs: int = 20,
    n_grid_repeats: int = 10,
    n_eval_runs: int = 100,
) -> Dict[str, Any]:
    """
    Executa todo o estudo downstream para um ticker:
      1) labels 3-classes
      2) grid search PRD
      3) grid search TE
      4) 100 execuções PRD
      5) 100 execuções TE
      6) 100 execuções Random
      7) salva JSONs de grid e avaliação

    Estrutura de saída:
      out_dir/grid_lstm/{ticker}_grid.json
      out_dir/MannWhitney_Data/{ticker}_eval.json
    """
    ensure_dir(out_dir / "grid_lstm")
    ensure_dir(out_dir / "MannWhitney_Data")

    # 1) labels com tau calculado só no treino
    labels_df, tau, H = build_labels_for_ticker_train_test(
        data_dir=data_dir,
        ticker=ticker,
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
    )

    # 2) representações
    prd_df = load_prd_representation_for_ticker(data_dir, ticker)
    te_df = load_te_representation_for_ticker(emb_csv, ticker)

    ae_df = None
    if ae_emb_csv is not None:
        ae_df = load_te_representation_for_ticker(ae_emb_csv, ticker)

    # 3) grid search
    grid = LstmGrid()

    print(f"[{ticker}] grid search PRD ...")
    best_prd = grid_search_lstm_for_representation(
        rep_df=prd_df,
        labels_df=labels_df,
        grid=grid,
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
        n_repeats_per_grid_point=n_grid_repeats,
        epochs=epochs,
        device=device,
    )

    print(f"[{ticker}] grid search TE ...")
    best_te = grid_search_lstm_for_representation(
        rep_df=te_df,
        labels_df=labels_df,
        grid=grid,
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
        n_repeats_per_grid_point=n_grid_repeats,
        epochs=epochs,
        device=device,
    )

    best_ae = None
    if ae_df is not None:
        print(f"[{ticker}] grid search CNN-AE ...")
        best_ae = grid_search_lstm_for_representation(
            rep_df=ae_df,
            labels_df=labels_df,
            grid=grid,
            train_end=train_end,
            test_start=test_start,
            test_end=test_end,
            n_repeats_per_grid_point=n_grid_repeats,
            epochs=epochs,
            device=device,
        )

    # 4) avaliações finais
    print(f"[{ticker}] 100 execuções PRD ...")
    acu_raw = evaluate_representation_100x(
        rep_df=prd_df,
        labels_df=labels_df,
        best_params=best_prd["best_params"],
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
        n_runs=n_eval_runs,
        epochs=epochs,
        device=device,
    )

    print(f"[{ticker}] 100 execuções TE ...")
    acu_te = evaluate_representation_100x(
        rep_df=te_df,
        labels_df=labels_df,
        best_params=best_te["best_params"],
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
        n_runs=n_eval_runs,
        epochs=epochs,
        device=device,
    )

    acu_ae = []
    if ae_df is not None and best_ae is not None and best_ae["best_params"] is not None:
        print(f"[{ticker}] 100 execuções CNN-AE ...")
        acu_ae = evaluate_representation_100x(
            rep_df=ae_df,
            labels_df=labels_df,
            best_params=best_ae["best_params"],
            train_end=train_end,
            test_start=test_start,
            test_end=test_end,
            n_runs=n_eval_runs,
            epochs=epochs,
            device=device,
        )

    # 5) random baseline
    # usa window=5 só para obter o mesmo y_test
    print(f"[{ticker}] 100 execuções RC ...")
    acu_rand = evaluate_random_classifier_100x(
        labels_df=labels_df,
        rep_df=prd_df,
        window=5,
        train_end=train_end,
        test_start=test_start,
        test_end=test_end,
        n_runs=n_eval_runs,
    )

    # 6) p-values (opcional)
    p_te_vs_rc = mannwhitney_one_tailed_greater(acu_te, acu_rand)
    p_te_vs_prd = mannwhitney_one_tailed_greater(acu_te, acu_raw)
    p_te_vs_ae = mannwhitney_one_tailed_greater(acu_te, acu_ae) if acu_ae else None

    # 7) salva JSON de grid
    grid_json = {
        "ticker": ticker,
        "can_execute": True,
        "tau_ticker": tau,
        "entropy": H,
        "best_params_raw": best_prd["best_params"],
        "best_params_encoder": best_ae["best_params"] if best_ae is not None else None,
        "best_params_encoder_adversarial": best_te["best_params"],
    }
    with open(out_dir / "grid_lstm" / f"{ticker}_grid.json", "w", encoding="utf-8") as f:
        json.dump(grid_json, f, indent=2)

    # 8) salva JSON de avaliação (mesma ideia do OSF)
    eval_json = {
        "ticker": ticker,
        "can_execute": True,
        "acu_rand": acu_rand,
        "acu_raw": acu_raw,
        "acu_encoder": acu_ae,
        "acu_encoder_adversarial": acu_te,
        "p_te_vs_rc": p_te_vs_rc,
        "p_te_vs_prd": p_te_vs_prd,
        "p_te_vs_ae": p_te_vs_ae,
        "mean_rand": float(np.mean(acu_rand)) if acu_rand else None,
        "mean_raw": float(np.mean(acu_raw)) if acu_raw else None,
        "mean_encoder": float(np.mean(acu_ae)) if acu_ae else None,
        "mean_encoder_adversarial": float(np.mean(acu_te)) if acu_te else None,
    }
    with open(out_dir / "MannWhitney_Data" / f"{ticker}_eval.json", "w", encoding="utf-8") as f:
        json.dump(eval_json, f, indent=2)

    return {
        "grid": grid_json,
        "eval": eval_json,
    }


# --------------------------------------------------------------------------
# COMANDO NOVO: lstmstudy
# --------------------------------------------------------------------------
def cmd_lstmstudy(args: argparse.Namespace) -> None:
    """
    Roda o estudo downstream LSTM.

    Se faltar data_79.npz para um ticker ainda não concluído, o ticker é marcado
    como 'missing_data' e o loop continua.

    Robustez adicional:
    - salva o summary CSV a cada ticker processado;
    - gera um ZIP incremental (rolling checkpoint) após cada ticker concluído com sucesso,
      sobrescrevendo sempre o mesmo arquivo parcial;
    - também registra o estado em caso de skip por missing_data ou erro, para manter
      o resumo e o progresso consistentes no Colab.
    """
    tickers = args.tickers or [t for t in DJIA_30 if t != "DOW"]
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    out_dir = Path(args.out_dir)
    ensure_dir(out_dir)
    ensure_dir(out_dir / "grid_lstm")
    ensure_dir(out_dir / "MannWhitney_Data")

    data_dir = Path(args.data_dir)
    exp_root = Path(args.exp_root).resolve() if getattr(args, "exp_root", "") else data_dir.resolve().parent
    incremental_bundle_name = getattr(args, "incremental_bundle_name", "after_lstmstudy_partial.zip")
    incremental_bundle_fp = BUNDLE_DIR / "bundles" / incremental_bundle_name
    incremental_bundle_fp.parent.mkdir(parents=True, exist_ok=True)

    summaries = []
    had_missing_or_error = False
    summary_fp = out_dir / "lstmstudy_summary.csv"

    def flush_summary_and_zip(reason: str, do_zip: bool) -> None:
        pd.DataFrame(summaries).to_csv(summary_fp, index=False)
        print(f"[OK] resumo parcial salvo em {summary_fp}")

        if do_zip:
            zip_experiment_state(exp_root, incremental_bundle_fp)
            print(f"[OK] checkpoint incremental ({reason}) salvo em {incremental_bundle_fp}")

    for tkr in tickers:
        print(f"\n==================== {tkr} ====================")

        grid_fp = out_dir / "grid_lstm" / f"{tkr}_grid.json"
        eval_fp = out_dir / "MannWhitney_Data" / f"{tkr}_eval.json"
        data_fp = data_dir / tkr / "data_79.npz"

        # 1) se já concluiu o ticker, pula
        if grid_fp.exists() and eval_fp.exists():
            print(f"[SKIP] {tkr} já concluído")
            summaries.append({
                "ticker": tkr,
                "status": "done_already",
            })
            flush_summary_and_zip(reason=f"{tkr}_done_already", do_zip=False)
            continue

        # 2) se não concluiu e não há dados brutos, marca como missing_data
        if not data_fp.exists():
            print(f"[SKIP MISSING DATA] {tkr}: {data_fp} não encontrado")
            summaries.append({
                "ticker": tkr,
                "status": "missing_data",
                "missing_file": str(data_fp),
            })
            had_missing_or_error = True
            flush_summary_and_zip(reason=f"{tkr}_missing_data", do_zip=True)
            continue

        # 3) tenta executar normalmente
        try:
            res = run_lstm_downstream_for_ticker(
                data_dir=data_dir,
                emb_csv=Path(args.emb_csv),
                ae_emb_csv=Path(args.ae_emb_csv) if args.ae_emb_csv else None,
                out_dir=out_dir,
                ticker=tkr,
                train_end=args.train_end,
                test_start=args.test_start,
                test_end=args.test_end,
                device=args.device,
                epochs=args.epochs,
                n_grid_repeats=args.n_grid_repeats,
                n_eval_runs=args.n_eval_runs,
            )

            summaries.append({
                "ticker": tkr,
                "status": "done",
                "mean_rand": res["eval"]["mean_rand"],
                "mean_raw": res["eval"]["mean_raw"],
                "mean_ae": res["eval"]["mean_encoder"],
                "mean_te": res["eval"]["mean_encoder_adversarial"],
                "p_te_vs_rc": res["eval"]["p_te_vs_rc"],
                "p_te_vs_prd": res["eval"]["p_te_vs_prd"],
                "p_te_vs_ae": res["eval"]["p_te_vs_ae"],
            })

            flush_summary_and_zip(reason=f"{tkr}_done", do_zip=True)

        except Exception as e:
            print(f"[ERRO] {tkr}: {e}")
            summaries.append({
                "ticker": tkr,
                "status": "error",
                "error": str(e),
            })
            had_missing_or_error = True
            flush_summary_and_zip(reason=f"{tkr}_error", do_zip=True)

    pd.DataFrame(summaries).to_csv(summary_fp, index=False)
    print(f"\n[OK] resumo salvo em {summary_fp}")

    # sinaliza se o estudo ficou parcial
    if had_missing_or_error:
        print("[WARN] lstmstudy terminou parcialmente (há tickers faltando ou com erro).")
    else:
        print("[OK] lstmstudy concluído integralmente.")

def lstmstudy_completed(exp_root: Path, expected_tickers: Optional[List[str]] = None) -> bool:
    fp = exp_root / "outputs" / "lstmstudy_mock" / "lstmstudy_summary.csv"
    if not fp.exists():
        return False

    df = pd.read_csv(fp)
    if df.empty:
        return False
    if "ticker" not in df.columns or "status" not in df.columns:
        return False

    # status que significam ticker concluído com sucesso
    success_status = {"ok", "done", "done_already"}

    # se quisermos validar contra a lista esperada de tickers
    if expected_tickers is not None:
        expected = {t.upper() for t in expected_tickers}
        got = {str(t).upper() for t in df["ticker"].dropna().tolist()}

        # se não cobriu todos os tickers esperados, ainda não terminou
        if got != expected:
            return False

        df = df[df["ticker"].astype(str).str.upper().isin(expected)].copy()

    return df["status"].astype(str).isin(success_status).all()
# ============================================================================
# COMANDOS
# ============================================================================
def cmd_searchae(args: argparse.Namespace) -> None:
    """
    Executa o hyperparameter search completo dos autoencoders, de forma resumable.

    model:
      te / ae / all
    """
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    X_all, meta = load_all_tickers_npz(
        Path(args.data_dir),
        tickers,
        date_start=args.date_start,
        date_end=args.date_end,
    )

    # fixa os hiperparâmetros “externos” conforme o artigo
    cfg = TrainConfig(
        seed=args.seed,
        device=args.device,
        epochs=args.epochs,
        batch_size=32,
        lr=1e-3,
        weight_decay=args.weight_decay,
        dropout=0.1,
        z_dim=64,
        lambda_recon=1.0,
        lambda_adv=1.0,
        angle_clip_deg=args.angle_clip_deg,
        num_workers=args.num_workers,
    )

    out_dir = Path(args.out_dir)
    ensure_dir(out_dir)

    if args.model in ("te", "all"):
        print("[searchae] iniciando busca TE-CNN-AAE")
        res_te = search_autoencoder_grid("te", X_all, meta, out_dir / "search_te", cfg)
        print("[searchae] melhor TE:", res_te["best_arch"], "score=", res_te["best_score"])

    if args.model in ("ae", "all"):
        print("[searchae] iniciando busca CNN-AE")
        res_ae = search_autoencoder_grid("ae", X_all, meta, out_dir / "search_ae", cfg)
        print("[searchae] melhor AE:", res_ae["best_arch"], "score=", res_ae["best_score"])

def cmd_mockdata(args: argparse.Namespace) -> None:
    """
    Executa a etapa de construção de dados sintéticos.

    Leitura mental:
      Passo 1: definir lista de tickers
      Passo 2: criar a fonte mock
      Passo 3: construir e salvar bundles padronizados
    """
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    source = MockDaily79Source(
        start_date=args.start_date,
        n_days=args.n_days,
        seed=args.seed,
    )

    build_from_source(source, tickers, Path(args.out_dir))


def cmd_csvdata(args: argparse.Namespace) -> None:
    """
    Executa a etapa de construção de dados reais a partir de CSV.

    Esta função é a substituta natural do mockdata quando você tiver os dados
    do orientador.

    Leitura mental:
      Passo 1: definir tickers
      Passo 2: criar a fonte CSV
      Passo 3: converter tudo para o schema padrão
    """
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    source = CsvDaily79Source(
        csv_dir=args.csv_dir,
        date_col=args.date_col,
        timestamp_col=args.timestamp_col,
        close_col=args.close_col,
        completeness_min=args.completeness_min,
        session_start_utc=args.session_start_utc,
        assume_timezone=args.assume_timezone,
        require_open_quote=args.require_open_quote,
    )

    build_from_source(source, tickers, Path(args.out_dir))

# ============================================================================
# LOADER GENÉRICO DE MODELO A PARTIR DE CHECKPOINT
# ============================================================================
#
# Esta função permite que o comando `embed` funcione tanto para:
# - TE-CNN-AAE
# - CNN-AE
#
# A decisão é feita pelo campo `model_type` salvo no checkpoint.
# ============================================================================

def load_model_from_checkpoint(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model_type = ckpt["model_type"]
    arch_dict = ckpt["arch_cfg"]

    arch = AeArchConfig(
        filters=tuple(arch_dict["filters"]),
        kernels=tuple(arch_dict["kernels"]),
        z_dim=int(arch_dict["z_dim"]),
        dropout=float(arch_dict["dropout"]),
    )

    if model_type == "te_cnn_aae":
        model = TeCnnAae(input_len=TARGET_LEN, arch=arch)
    elif model_type == "cnn_ae":
        model = CnnAe(input_len=TARGET_LEN, arch=arch)
    else:
        raise ValueError(f"model_type desconhecido: {model_type}")

    model.load_state_dict(ckpt["state_dict"], strict=True)
    return model, ckpt

def load_arch_cfg_from_json(json_path: Path) -> AeArchConfig:
    """
    Lê um JSON salvo por searchae e reconstrói o AeArchConfig.

    Aceita:
      - best_arch
      - best_metrics["arch"]
    """


    with open(json_path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if "best_arch" in obj and obj["best_arch"] is not None:
        arch_dict = obj["best_arch"]
    elif "best_metrics" in obj and obj["best_metrics"] is not None:
        arch_dict = obj["best_metrics"]["arch"]
    else:
        raise ValueError(f"Não foi possível localizar arquitetura em {json_path}")

    return AeArchConfig(
        filters=tuple(arch_dict["filters"]),
        kernels=tuple(arch_dict["kernels"]),
        z_dim=int(arch_dict["z_dim"]),
        dropout=float(arch_dict["dropout"]),
    )

def cmd_train(args: argparse.Namespace) -> None:
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    X_all, meta = load_all_tickers_npz(
        Path(args.data_dir),
        tickers,
        date_start=args.date_start,
        date_end=args.date_end,
    )
    print(f"[train] X shape = {X_all.shape}")

    cfg = TrainConfig(
        seed=args.seed,
        device=args.device,
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        weight_decay=args.weight_decay,
        dropout=args.dropout,
        z_dim=args.z_dim,
        lambda_recon=args.lambda_recon,
        lambda_adv=args.lambda_adv,
        angle_clip_deg=args.angle_clip_deg,
        num_workers=args.num_workers,
    )

    arch_cfg = None
    if args.arch_json:
        arch_cfg = load_arch_cfg_from_json(Path(args.arch_json))
        print("[train] usando arch do JSON:", arch_cfg.as_dict())

    train_tecnn_aae(X_all, meta, cfg, Path(args.model_out), arch_cfg=arch_cfg)




def cmd_embed(args: argparse.Namespace) -> None:
    """
    Executa a extração de embeddings com um modelo já treinado.

    Compatível com:
      - TE-CNN-AAE
      - CNN-AE

    Leitura mental:
      Passo 1: carregar checkpoint
      Passo 2: reconstruir automaticamente a arquitetura correta
      Passo 3: carregar os dados
      Passo 4: extrair embeddings z
      Passo 5: salvar em CSV
      Passo 6: opcionalmente salvar em ARFF
    """
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    # 1) carrega o modelo correto a partir do checkpoint
    model, _ = load_model_from_checkpoint(Path(args.model))

    # 2) carrega os dados
    X_all, meta = load_all_tickers_npz(Path(args.data_dir), tickers)

    # 3) extrai embeddings
    Z = extract_embeddings(model, X_all, device=args.device, batch_size=args.batch_size)

    # 4) salva CSV
    out_csv = Path(args.out_csv)
    ensure_dir(out_csv.parent)

    df = pd.DataFrame(meta)
    for j in range(Z.shape[1]):
        df[f"z{j}"] = Z[:, j].astype(np.float32)
    df.to_csv(out_csv, index=False)
    print(f"[OK] CSV salvo em {out_csv}")

    # 5) salva ARFF opcional
    if args.out_arff:
        out_arff = Path(args.out_arff)
        ensure_dir(out_arff.parent)
        write_arff_embeddings(out_arff, Z)
        print(f"[OK] ARFF salvo em {out_arff}")
# ============================================================================
# ANÁLISE DE QUALIDADE DOS EMBEDDINGS (UMAP + SILHOUETTE + DAVIES-BOULDIN)
# ============================================================================

def cmd_analyze(args: argparse.Namespace) -> None:
    """
    Gera análise qualitativa e quantitativa dos embeddings:
      - UMAP 2D para visualização
      - Silhouette Score
      - Davies-Bouldin Index

    Replica a Tabela III e as Figuras 4a/4b/4c do artigo.

    Fluxo:
      1) carrega embeddings CSV (gerado por cmd_embed)
      2) recomputa ângulos a partir de X_norm para montar os labels de tendência
      3) aplica UMAP
      4) calcula Silhouette e Davies-Bouldin
      5) salva plot PNG e metrics.json
    """
    try:
        import umap
        from sklearn.metrics import silhouette_score, davies_bouldin_score
        import matplotlib.pyplot as plt
    except ImportError as e:
        print(f"[ERRO] Dependência faltando: {e}")
        print("Instale com: pip install umap-learn scikit-learn matplotlib")
        return

    out_dir = Path(args.out_dir)
    ensure_dir(out_dir)

    # ------------------------------------------------------------------
    # 1) Carrega embeddings
    # ------------------------------------------------------------------
    df_emb = pd.read_csv(args.emb_csv)

    if args.date_start:
        df_emb = df_emb[df_emb["date"].astype(str) >= str(args.date_start)].copy()
    if args.date_end:
        df_emb = df_emb[df_emb["date"].astype(str) <= str(args.date_end)].copy()

    z_cols = [c for c in df_emb.columns if c.startswith("z")]
    if not z_cols:
        raise RuntimeError("Nenhuma coluna z* encontrada no CSV de embeddings.")

    if "ticker" not in df_emb.columns or "date" not in df_emb.columns:
        raise RuntimeError("O CSV de embeddings precisa ter colunas 'ticker' e 'date'.")

    df_emb = df_emb.copy()
    df_emb["ticker"] = df_emb["ticker"].astype(str)
    df_emb["date"] = df_emb["date"].astype(str)
    df_emb["_row_id"] = np.arange(len(df_emb))

    # ------------------------------------------------------------------
    # 2) Recomputa ângulos para montar os labels de tendência
    #    (mesma lógica do artigo: regressão linear sobre X_norm)
    # ------------------------------------------------------------------
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    X_all, meta = load_all_tickers_npz(
        Path(args.data_dir),
        tickers,
        date_start=args.date_start,
        date_end=args.date_end,
    )

    t_vec = np.arange(TARGET_LEN, dtype=np.float64)
    mean_t = t_vec.mean()
    var_t = ((t_vec - mean_t) ** 2).sum()

    angles = []
    for row in X_all:
        x = row.astype(np.float64)
        m = ((t_vec - mean_t) * (x - x.mean())).sum() / var_t
        angle_deg = float(np.degrees(np.arctan(m)))
        angle_deg = float(np.clip(angle_deg, -1.0, 1.0))
        angles.append(angle_deg)

    df_angle = pd.DataFrame(meta)[["ticker", "date"]].copy()
    df_angle["ticker"] = df_angle["ticker"].astype(str)
    df_angle["date"] = df_angle["date"].astype(str)
    df_angle["angle_deg"] = np.asarray(angles, dtype=np.float32)

    df_work = df_emb.merge(
        df_angle,
        on=["ticker", "date"],
        how="inner",
        validate="one_to_one",
    )

    df_work = df_work.sort_values("_row_id").reset_index(drop=True)

    if len(df_work) != len(df_emb):
        raise RuntimeError(
            f"[analyze] desalinhamento entre embeddings e ângulos: "
            f"{len(df_emb)} linhas no CSV, {len(df_work)} após merge."
        )

    Z = df_work[z_cols].to_numpy(dtype=np.float32)
    print(f"[analyze] embeddings carregados: {Z.shape}")

    # Mesmos 5 bins do artigo
    bins = [-1.0, -0.6, -0.2, 0.2, 0.6, 1.0]
    labels_str = ["-1° a -0.6°", "-0.6° a -0.2°", "-0.2° a 0.2°",
                  "0.2° a 0.6°", "0.6° a 1°"]
    angle_labels = np.digitize(
        df_work["angle_deg"].to_numpy(dtype=np.float32),
        bins[1:-1]
    )  # 0..4

    # ------------------------------------------------------------------
    # 3) UMAP
    # ------------------------------------------------------------------
    print("[analyze] rodando UMAP ...")
    reducer  = umap.UMAP(n_components=2, random_state=42)
    Z_2d     = reducer.fit_transform(Z)

    # ------------------------------------------------------------------
    # 4) Métricas de cluster
    # ------------------------------------------------------------------
    ss = silhouette_score(Z, angle_labels)
    db = davies_bouldin_score(Z, angle_labels)
    print(f"[analyze] Silhouette Score : {ss:.4f}")
    print(f"[analyze] Davies-Bouldin   : {db:.4f}")

    # ------------------------------------------------------------------
    # 5) Salva métricas em JSON
    # ------------------------------------------------------------------
    metrics = {
        "emb_csv":        str(args.emb_csv),
        "silhouette":     round(float(ss), 6),
        "davies_bouldin": round(float(db), 6),
        "n_samples":      int(Z.shape[0]),
        "n_dims":         int(Z.shape[1]),
    }
    metrics_fp = out_dir / "metrics.json"
    with open(metrics_fp, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"[OK] métricas salvas em {metrics_fp}")

    # ------------------------------------------------------------------
    # 6) Salva plot UMAP
    # ------------------------------------------------------------------
    colors = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4"]
    _, ax = plt.subplots(figsize=(7, 5))
    for cls in range(5):
        mask = angle_labels == cls
        ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                   c=colors[cls], label=labels_str[cls],
                   s=6, alpha=0.6)
    ax.legend(markerscale=2, fontsize=8)
    model_name = Path(args.emb_csv).stem
    ax.set_title(f"UMAP — {model_name}\nSilhouette={ss:.4f}  Davies-Bouldin={db:.4f}")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    plt.tight_layout()

    plot_fp = out_dir / f"umap_{model_name}.png"
    plt.savefig(plot_fp, dpi=150)
    plt.close()
    print(f"[OK] plot salvo em {plot_fp}")
# --------------------------------------------------------------------------
# RESUMO DA GERAÇÃO DE LABELS 3-CLASSES
# --------------------------------------------------------------------------
# Esta etapa NÃO usa o TE-CNN-AAE diretamente.
#
# Ela usa:
#   eod_close_raw -> delta diário -> threshold por entropia máxima -> labels
#
# Portanto:
#   labels é um ramo paralelo ao embedding, e não uma saída do autoencoder.
# --------------------------------------------------------------------------
def cmd_labels(args: argparse.Namespace) -> None:
    """
    Gera os rótulos 3-classes por ticker usando o fechamento bruto diário,
    calculando tau SOMENTE no treino.
    """
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    rows = []
    data_dir = Path(args.data_dir)
    names = np.array(["Decrease", "NoAction", "Increase"], dtype=object)

    for tkr in tickers:
        fp = data_dir / tkr / "data_79.npz"
        if not fp.exists():
            continue

        try:
            df_lbl, tau, H = build_labels_for_ticker_train_test(
                data_dir=data_dir,
                ticker=tkr,
                train_end=args.train_end,
                test_start=args.test_start,
                test_end=args.test_end,
            )
        except RuntimeError as e:
            print(f"[WARN] {e}")
            continue

        dates, eod = load_per_ticker_close_series(data_dir, tkr)
        label_dates = np.asarray(dates[1:], dtype=str)
        eod_label = np.asarray(eod[1:], dtype=np.float64)

        if len(df_lbl) != len(label_dates):
            raise RuntimeError(
                f"[labels] desalinhamento em {tkr}: {len(df_lbl)} labels vs {len(label_dates)} datas."
            )

        for i, row in df_lbl.reset_index(drop=True).iterrows():
            rows.append({
                "ticker": tkr,
                "date": str(label_dates[i]),
                "eod_close_raw": float(eod_label[i]),
                "delta": float(row["delta"]),
                "tau_ticker": float(tau),
                "entropy": float(H),
                "label": int(row["label"]),
                "label_name": str(names[int(row["label"])]),
                "split": str(row["split"]),
            })

    out_csv = Path(args.out_csv)
    ensure_dir(out_csv.parent)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[OK] labels salvos em {out_csv}")

def cmd_trainae(args: argparse.Namespace) -> None:
    tickers = args.tickers or DJIA_30
    if isinstance(tickers, str):
        tickers = [t.strip().upper() for t in tickers.split(",") if t.strip()]

    X_all, meta = load_all_tickers_npz(
        Path(args.data_dir),
        tickers,
        date_start=args.date_start,
        date_end=args.date_end,
    )
    print(f"[trainae] X shape = {X_all.shape}")

    cfg = TrainConfig(
        seed=args.seed,
        device=args.device,
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        weight_decay=args.weight_decay,
        dropout=args.dropout,
        z_dim=args.z_dim,
        lambda_recon=args.lambda_recon,
        lambda_adv=args.lambda_adv,
        angle_clip_deg=args.angle_clip_deg,
        num_workers=args.num_workers,
    )

    arch_cfg = None
    if args.arch_json:
        arch_cfg = load_arch_cfg_from_json(Path(args.arch_json))
        print("[trainae] usando arch do JSON:", arch_cfg.as_dict())

    train_cnn_ae(X_all, meta, cfg, Path(args.model_out), arch_cfg=arch_cfg)
# ============================================================================
# PARSER DE ARGUMENTOS
# ============================================================================

def build_parser() -> argparse.ArgumentParser:
    """
    Monta o parser de subcomandos.

    Subcomandos disponíveis:
    - mockdata
    - csvdata
    - train
    - embed
    - labels
    - lstmstudy
    """
    p = argparse.ArgumentParser(
        "te_cnn_aae_modular.py",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    sub = p.add_subparsers(dest="cmd", required=True)

    # mockdata
    p_mk = sub.add_parser("mockdata", help="Gerar dataset sintético")
    p_mk.add_argument("--out-dir", required=True)
    p_mk.add_argument("--tickers", default="")
    p_mk.add_argument("--start-date", default="2018-01-01")
    p_mk.add_argument("--n-days", type=int, default=800)
    p_mk.add_argument("--seed", type=int, default=123)
    p_mk.set_defaults(func=cmd_mockdata)

    # csvdata
    p_csv = sub.add_parser("csvdata", help="Converter CSV para schema padrão data_79.npz")
    p_csv.add_argument("--csv-dir", required=True, help="Pasta contendo {TICKER}.csv")
    p_csv.add_argument("--out-dir", required=True)
    p_csv.add_argument("--tickers", default="")
    p_csv.add_argument("--date-col", default="date")
    p_csv.add_argument("--timestamp-col", default="timestamp")
    p_csv.add_argument("--close-col", default="close")
    p_csv.add_argument("--completeness-min", type=float, default=0.80)
    p_csv.add_argument("--session-start-utc", default="14:30")
    p_csv.add_argument("--assume-timezone", default="UTC")
    p_csv.add_argument(
        "--require-open-quote",
        dest="require_open_quote",
        action="store_true",
        help="Exigir a observação da abertura da sessão (14:30 UTC)."
    )
    p_csv.add_argument(
        "--allow-missing-open-quote",
        dest="require_open_quote",
        action="store_false",
        help="Permitir dias sem a observação de abertura. Use apenas fora do protocolo do paper."
    )
    p_csv.set_defaults(func=cmd_csvdata, require_open_quote=True)


    # searchae
    p_sa = sub.add_parser("searchae", help="Hyperparameter search completo dos autoencoders")
    p_sa.add_argument("--data-dir", required=True)
    p_sa.add_argument("--out-dir", required=True)
    p_sa.add_argument("--model", default="all", choices=["te", "ae", "all"])
    p_sa.add_argument("--tickers", default="")
    p_sa.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))
    p_sa.add_argument("--seed", type=int, default=123)
    p_sa.add_argument("--epochs", type=int, default=20)
    p_sa.add_argument("--weight-decay", type=float, default=0.0)
    p_sa.add_argument("--angle-clip-deg", type=float, default=1.0)
    p_sa.add_argument("--num-workers", type=int, default=0)
    p_sa.add_argument("--date-start", default="2018-01-01")
    p_sa.add_argument("--date-end", default="2022-12-31")
    p_sa.set_defaults(func=cmd_searchae)

    # trainae
    p_tae = sub.add_parser("trainae", help="Treinar CNN-AE (ablação sem Trend Discriminator)")
    p_tae.add_argument("--data-dir", required=True)
    p_tae.add_argument("--model-out", required=True)
    p_tae.add_argument("--tickers", default="")
    p_tae.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))
    p_tae.add_argument("--seed", type=int, default=123)
    p_tae.add_argument("--epochs", type=int, default=20)
    p_tae.add_argument("--batch-size", type=int, default=64)
    p_tae.add_argument("--lr", type=float, default=1e-3)
    p_tae.add_argument("--weight-decay", type=float, default=0.0)
    p_tae.add_argument("--dropout", type=float, default=0.1)
    p_tae.add_argument("--z-dim", type=int, default=64)
    p_tae.add_argument("--lambda-recon", type=float, default=1.0)
    p_tae.add_argument("--lambda-adv", type=float, default=0.0)
    p_tae.add_argument("--angle-clip-deg", type=float, default=1.0)
    p_tae.add_argument("--num-workers", type=int, default=0)
    p_tae.add_argument("--arch-json", default="", help="JSON com a melhor arquitetura do CNN-AE encontrada no searchae")
    p_tae.add_argument("--date-start", default="2018-01-01")
    p_tae.add_argument("--date-end", default="2022-12-31")
    p_tae.set_defaults(func=cmd_trainae)

    # train
    p_tr = sub.add_parser("train", help="Treinar TE-CNN-AAE")
    p_tr.add_argument("--data-dir", required=True)
    p_tr.add_argument("--model-out", required=True)
    p_tr.add_argument("--tickers", default="")
    p_tr.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))
    p_tr.add_argument("--seed", type=int, default=123)
    p_tr.add_argument("--epochs", type=int, default=20)
    p_tr.add_argument("--batch-size", type=int, default=64)
    p_tr.add_argument("--lr", type=float, default=1e-3)
    p_tr.add_argument("--weight-decay", type=float, default=0.0)
    p_tr.add_argument("--dropout", type=float, default=0.1)
    p_tr.add_argument("--z-dim", type=int, default=64)
    p_tr.add_argument("--lambda-recon", type=float, default=1.0)
    p_tr.add_argument("--lambda-adv", type=float, default=1.0)
    p_tr.add_argument("--angle-clip-deg", type=float, default=1.0)
    p_tr.add_argument("--num-workers", type=int, default=0)
    p_tr.add_argument("--arch-json", default="", help="JSON com a melhor arquitetura do TE-CNN-AAE encontrada no searchae")
    p_tr.add_argument("--date-start", default="2018-01-01")
    p_tr.add_argument("--date-end", default="2022-12-31")
    p_tr.set_defaults(func=cmd_train)

    # embed
    p_em = sub.add_parser("embed", help="Extrair embeddings")
    p_em.add_argument("--data-dir", required=True)
    p_em.add_argument("--model", required=True)
    p_em.add_argument("--out-csv", required=True)
    p_em.add_argument("--out-arff", default="")
    p_em.add_argument("--tickers", default="")
    p_em.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))
    p_em.add_argument("--batch-size", type=int, default=256)
    p_em.add_argument("--z-dim", type=int, default=64)
    p_em.set_defaults(func=cmd_embed)


    # analyze
    p_an = sub.add_parser("analyze", help="UMAP + Silhouette + Davies-Bouldin nos embeddings")
    p_an.add_argument("--emb-csv",  required=True, help="CSV de embeddings gerado por cmd_embed")
    p_an.add_argument("--data-dir", required=True, help="Pasta com {ticker}/data_79.npz")
    p_an.add_argument("--out-dir",  required=True, help="Pasta de saída (PNG + metrics.json)")
    p_an.add_argument("--tickers",  default="")
    p_an.add_argument("--date-start", default="2018-01-01")
    p_an.add_argument("--date-end", default="2022-12-31")
    p_an.set_defaults(func=cmd_analyze)


    # labels
    p_lb = sub.add_parser("labels", help="Gerar labels 3-classes por entropia máxima")
    p_lb.add_argument("--data-dir", required=True)
    p_lb.add_argument("--out-csv", required=True)
    p_lb.add_argument("--tickers", default="")
    p_lb.add_argument("--train-end", default="2022-12-31")
    p_lb.add_argument("--test-start", default="2023-01-01")
    p_lb.add_argument("--test-end", default="2023-01-31")
    p_lb.set_defaults(func=cmd_labels)

    # lstmstudy
    p_ls = sub.add_parser("lstmstudy", help="Executar estudo downstream LSTM (PRD + TE + RC)")
    p_ls.add_argument("--data-dir", required=True, help="Pasta com {ticker}/data_79.npz")
    p_ls.add_argument("--emb-csv", required=True, help="CSV de embeddings gerado por cmd_embed")
    p_ls.add_argument("--out-dir", required=True, help="Pasta de saída dos resultados downstream")
    p_ls.add_argument("--tickers", default="", help="Lista de tickers separada por vírgula")
    p_ls.add_argument("--train-end", default="2022-12-31")
    p_ls.add_argument("--test-start", default="2023-01-01")
    p_ls.add_argument("--test-end", default="2023-01-31")
    p_ls.add_argument("--device", default=("cuda" if torch.cuda.is_available() else "cpu"))
    p_ls.add_argument("--epochs", type=int, default=20)
    p_ls.add_argument("--n-grid-repeats", type=int, default=10)
    p_ls.add_argument("--n-eval-runs", type=int, default=100)
    p_ls.add_argument("--ae-emb-csv", default="", help="CSV de embeddings do CNN-AE (ablação)")
    p_ls.add_argument("--exp-root", default="", help="Raiz do experimento para zip incremental do lstmstudy")
    p_ls.add_argument("--incremental-bundle-name", default="after_lstmstudy_partial.zip",
                      help="Nome do ZIP incremental sobrescrito após cada ticker")
    p_ls.set_defaults(func=cmd_lstmstudy)

    return p


# ============================================================================
# MAIN
# ============================================================================

def main(argv=None):
    """
    Ponto de entrada principal.

    NO NOTEBOOK/COLAB
    -----------------
    O uso recomendado é chamar explicitamente:

        main(["mockdata", "--out-dir", "./djia_mock"])
        main(["train", "--data-dir", "./djia_mock", "--model-out", "./models/m.pt"])
        main(["embed", "--data-dir", "./djia_mock", "--model", "./models/m.pt", "--out-csv", "./z.csv"])

    Isso evita o problema de o argparse tentar interpretar os argumentos
    internos do kernel do Jupyter.

    LEITURA MENTAL
    --------------
    Passo 1: montar o parser
    Passo 2: interpretar argv
    Passo 3: completar tickers padrão se necessário
    Passo 4: despachar para a função do subcomando
    """
    parser = build_parser()

    # Passo 1: converter argv em objeto args
    args = parser.parse_args(argv)

    # Passo 2: se o usuário não informou tickers, usar a lista padrão
    if hasattr(args, "tickers") and isinstance(args.tickers, str) and args.tickers.strip() == "":
        args.tickers = DJIA_30

    # Passo 3: executar o subcomando escolhido
    args.func(args)





# ============================================================================
# RUNNERS DE EXPERIMENTO
# ============================================================================




def run_mock_pipeline_tecnn_protocol(
    exp_root: str = "./exp_tecnn_mock_protocol",
    restore_zips: bool = False,
    enable_bundles: bool = True,
    mock_start_date: str = "2018-01-01",
    mock_n_days: int = 1350,
    mock_seed: int = 123,
    train_start: str = "2018-01-01",
    train_end: str = "2022-12-31",
    test_start: str = "2023-01-01",
    test_end: str = "2023-01-31",
    search_epochs: int = 20,
    train_epochs: int = 20,
    ae_epochs: int = 20,
    lstm_epochs: int = 20,
    n_grid_repeats: int = 10,
    n_eval_runs: int = 100,
):
    exp_root = Path(exp_root)
    ensure_exp_dirs(exp_root)

    def _zip(bundle_name: str) -> None:
        if enable_bundles:
            zip_experiment_state(exp_root, BUNDLE_DIR / "bundles" / bundle_name)

    if restore_zips:
        search_dirs = [
            Path("/content"),
            BUNDLE_DIR / "bundles",
            exp_root / "bundles",
        ]
        zips = cleanup_duplicate_zips(
            search_dirs=search_dirs,
            quarantine_dir=Path("/content/_zip_duplicates"),
            dry_run=False,
        )
        print("Encontrados após limpeza:", zips)

        if zips:
            merged_manifest = restore_many_experiment_states(
                zip_paths=zips,
                exp_root=exp_root,
            )
            print("[OK] Restauração consolidada concluída.")
            print("Manifest consolidado:", json.dumps(merged_manifest, indent=2, ensure_ascii=False))
        else:
            print("[INFO] Nenhum ZIP válido encontrado para restauração.")
    else:
        print("[INFO] Restauração automática de ZIPs desabilitada.")

    manifest = load_manifest(exp_root)
    done = stage_outputs(exp_root)

    manifest.setdefault("experiment_id", "exp_tecnn_mock_protocol")
    manifest.setdefault("code_version", "tecnn_stateful_v8_mock_protocol")
    manifest.setdefault("data_mode", "mock")
    manifest.setdefault("mock_generator_version", "mock_vstable_md5")
    manifest.setdefault("mock_seed_mode", "stable_md5_per_ticker")
    manifest.setdefault("mock_start_date", mock_start_date)
    manifest.setdefault("mock_n_days", mock_n_days)
    manifest.setdefault("mock_seed", mock_seed)
    manifest.setdefault("train_start", train_start)
    manifest.setdefault("train_end", train_end)
    manifest.setdefault("test_start", test_start)
    manifest.setdefault("test_end", test_end)
    manifest.setdefault("arch_version_te", "tecnn_v2_tconv_decoder")
    manifest.setdefault("arch_version_ae", "cnn_ae_table2b_v2")
    manifest.setdefault("te_hsearch_done", False)
    manifest.setdefault("ae_hsearch_done", False)
    manifest.setdefault("te_best_arch", None)
    manifest.setdefault("ae_best_arch", None)
    save_manifest(exp_root, manifest)

    # 1) mockdata
    if not done["mockdata"]:
        main([
            "mockdata",
            "--out-dir", str(exp_root / "data"),
            "--start-date", mock_start_date,
            "--n-days", str(mock_n_days),
            "--seed", str(mock_seed),
        ])
        manifest["mockdata_done"] = True
        save_manifest(exp_root, manifest)
        _zip("after_mockdata.zip")
    else:
        print("[OK] mockdata já foi executado.")

    # 2) searchae
    if not done["te_hsearch"] or not done["ae_hsearch"]:
        main([
            "searchae",
            "--data-dir", str(exp_root / "data"),
            "--out-dir", str(exp_root / "search_autoencoders"),
            "--model", "all",
            "--date-start", train_start,
            "--date-end", train_end,
            "--epochs", str(search_epochs),
        ])

        te_best_json = exp_root / "search_autoencoders" / "search_te" / "best_te.json"
        ae_best_json = exp_root / "search_autoencoders" / "search_ae" / "best_ae.json"

        manifest = load_manifest(exp_root)
        manifest["te_hsearch_done"] = te_best_json.exists()
        manifest["ae_hsearch_done"] = ae_best_json.exists()

        if te_best_json.exists():
            with open(te_best_json, "r", encoding="utf-8") as f:
                manifest["te_best_arch"] = json.load(f).get("best_arch")

        if ae_best_json.exists():
            with open(ae_best_json, "r", encoding="utf-8") as f:
                manifest["ae_best_arch"] = json.load(f).get("best_arch")

        save_manifest(exp_root, manifest)

        if manifest["te_hsearch_done"]:
            _zip("after_te_hsearch_partial.zip")
            _zip("after_te_hsearch.zip")

        if manifest["ae_hsearch_done"]:
            _zip("after_ae_hsearch_partial.zip")
            _zip("after_ae_hsearch.zip")
    else:
        print("[OK] hyperparameter search dos autoencoders já foi executado.")

    # 3) train TE-CNN-AAE
    if not done["te_train"]:
        main([
            "train",
            "--data-dir", str(exp_root / "data"),
            "--model-out", str(exp_root / "models" / "tecnn_aae_mock.pt"),
            "--arch-json", str(exp_root / "search_autoencoders" / "search_te" / "best_te.json"),
            "--date-start", train_start,
            "--date-end", train_end,
            "--epochs", str(train_epochs),
            "--batch-size", "32",
            "--lr", "0.001",
            "--dropout", "0.1",
            "--z-dim", "64",
            "--lambda-recon", "1.0",
            "--lambda-adv", "1.0",
        ])
        manifest["te_train_done"] = True
        save_manifest(exp_root, manifest)
        _zip("after_te_train.zip")
    else:
        print("[OK] train TE já foi executado.")

    # 4) embed TE-CNN-AAE
    if not done["te_embed"]:
        main([
            "embed",
            "--data-dir", str(exp_root / "data"),
            "--model", str(exp_root / "models" / "tecnn_aae_mock.pt"),
            "--out-csv", str(exp_root / "outputs" / "embeddings_tecnn_mock.csv"),
            "--out-arff", str(exp_root / "outputs" / "embeddings_tecnn_mock.arff"),
            "--batch-size", "256",
        ])
        manifest["te_embed_done"] = True
        save_manifest(exp_root, manifest)
        _zip("after_te_embed.zip")
    else:
        print("[OK] embed TE já foi executado.")

    # 5) train CNN-AE
    if not done["ae_train"]:
        main([
            "trainae",
            "--data-dir", str(exp_root / "data"),
            "--model-out", str(exp_root / "models" / "cnn_ae_mock.pt"),
            "--arch-json", str(exp_root / "search_autoencoders" / "search_ae" / "best_ae.json"),
            "--date-start", train_start,
            "--date-end", train_end,
            "--epochs", str(ae_epochs),
            "--batch-size", "32",
            "--lr", "0.001",
            "--dropout", "0.1",
            "--z-dim", "64",
        ])
        manifest["ae_train_done"] = True
        save_manifest(exp_root, manifest)
        _zip("after_ae_train.zip")
    else:
        print("[OK] train AE já foi executado.")

    # 6) embed CNN-AE
    if not done["ae_embed"]:
        main([
            "embed",
            "--data-dir", str(exp_root / "data"),
            "--model", str(exp_root / "models" / "cnn_ae_mock.pt"),
            "--out-csv", str(exp_root / "outputs" / "embeddings_cnnae_mock.csv"),
            "--out-arff", str(exp_root / "outputs" / "embeddings_cnnae_mock.arff"),
            "--batch-size", "256",
        ])
        manifest["ae_embed_done"] = True
        save_manifest(exp_root, manifest)
        _zip("after_ae_embed.zip")
    else:
        print("[OK] embed AE já foi executado.")

    # 7) analyze
    if not done["analyze"]:
        main([
            "analyze",
            "--emb-csv", str(exp_root / "outputs" / "embeddings_tecnn_mock.csv"),
            "--data-dir", str(exp_root / "data"),
            "--out-dir", str(exp_root / "outputs" / "analyze" / "tecnn"),
            "--tickers", ",".join(DJIA_30),
            "--date-start", train_start,
            "--date-end", train_end,
        ])

        main([
            "analyze",
            "--emb-csv", str(exp_root / "outputs" / "embeddings_cnnae_mock.csv"),
            "--data-dir", str(exp_root / "data"),
            "--out-dir", str(exp_root / "outputs" / "analyze" / "cnnae"),
            "--tickers", ",".join(DJIA_30),
            "--date-start", train_start,
            "--date-end", train_end,
        ])

        manifest = load_manifest(exp_root)
        manifest["analyze_done"] = (
            (exp_root / "outputs" / "analyze" / "tecnn" / "metrics.json").exists()
            and
            (exp_root / "outputs" / "analyze" / "cnnae" / "metrics.json").exists()
        )
        save_manifest(exp_root, manifest)

        if manifest["analyze_done"]:
            _zip("after_analyze.zip")
    else:
        print("[OK] analyze já foi executado.")

    # 8) labels
    if not done["labels"]:
        main([
            "labels",
            "--data-dir", str(exp_root / "data"),
            "--out-csv", str(exp_root / "outputs" / "labels_mock.csv"),
            "--train-end", train_end,
            "--test-start", test_start,
            "--test-end", test_end,
        ])
        manifest["labels_done"] = True
        save_manifest(exp_root, manifest)
        _zip("after_labels.zip")
    else:
        print("[OK] labels já foram gerados.")

    # 9) lstmstudy
    if not done["lstmstudy"]:
        main([
            "lstmstudy",
            "--data-dir", str(exp_root / "data"),
            "--emb-csv", str(exp_root / "outputs" / "embeddings_tecnn_mock.csv"),
            "--ae-emb-csv", str(exp_root / "outputs" / "embeddings_cnnae_mock.csv"),
            "--out-dir", str(exp_root / "outputs" / "lstmstudy_mock"),
            "--tickers", ",".join([t for t in DJIA_30 if t != "DOW"]),
            "--train-end", train_end,
            "--test-start", test_start,
            "--test-end", test_end,
            "--epochs", str(lstm_epochs),
            "--n-grid-repeats", str(n_grid_repeats),
            "--n-eval-runs", str(n_eval_runs),
            "--exp-root", str(exp_root),
            "--incremental-bundle-name", "after_lstmstudy_partial.zip",
        ])

        manifest = load_manifest(exp_root)
        expected_lstm_tickers = [t for t in DJIA_30 if t != "DOW"]
        manifest["lstmstudy_done"] = lstmstudy_completed(
            exp_root,
            expected_tickers=expected_lstm_tickers,
        )
        save_manifest(exp_root, manifest)

        if manifest["lstmstudy_done"]:
            _zip("after_lstmstudy.zip")
    else:
        print("[OK] lstmstudy já foi executado.")

    summary_fp = exp_root / "outputs" / "lstmstudy_mock" / "lstmstudy_summary.csv"
    if summary_fp.exists():
        df_sum = pd.read_csv(summary_fp)
        if "status" in df_sum.columns and df_sum["status"].isin(["missing_data", "error"]).any():
            print("[WARN] experimento concluído parcialmente: há tickers sem dados ou com erro.")
        else:
            print("[OK] experimento concluído com sucesso.")
    else:
        print("[WARN] experimento finalizado sem resumo downstream.")

    return exp_root
# ============================================================================
# PONTO DE ENTRADA MANUAL
# ============================================================================

# Escolha UMA opção para rodar:

# 1) smoke test completo com mock
# run_mock_pipeline_minimo()

# 2) OU, se preferir, uma configuração mais próxima da parte autoencoder do paper:
# run_mock_pipeline_artigo_parcial()



AUTO_RESTORE_ZIPS = True
AUTO_RUN_MOCK_RESUME = True

search_dirs = [
    Path("/content"),
    Path("/content/bundles_tecnn/bundles"),
]

if AUTO_RESTORE_ZIPS:
    zips = cleanup_duplicate_zips(
        search_dirs=search_dirs,
        quarantine_dir=Path("/content/_zip_duplicates"),
        dry_run=False,
    )

    print("Encontrados após limpeza:", zips)

    if zips:
        merged_manifest = restore_many_experiment_states(
            zip_paths=zips,
            exp_root=Path("./exp_tecnn")
        )
        print("[OK] Restauração consolidada concluída.")
        print("Manifest consolidado:", json.dumps(merged_manifest, indent=2, ensure_ascii=False))
    else:
        print("[INFO] Nenhum ZIP válido encontrado para restauração.")
else:
    print("[INFO] Restauração automática de ZIPs desabilitada por padrão.")

if AUTO_RUN_MOCK_RESUME:
    #run_mock_pipeline_resumable()
    run_mock_pipeline_tecnn_protocol(
        exp_root="./exp_tecnn_mock_protocol",
        restore_zips=True,
        enable_bundles=True,
    )
else:
    print("[INFO] Runner mock desabilitado por padrão.")
    print("[INFO] Para reproduzir com dados reais, execute explicitamente as etapas com main([...]).")

[INFO] Nenhum ZIP encontrado para limpeza.
Encontrados após limpeza: []
[INFO] Nenhum ZIP válido encontrado para restauração.
[INFO] Nenhum ZIP encontrado para limpeza.
Encontrados após limpeza: []
[INFO] Nenhum ZIP válido encontrado para restauração.
[OK] AAPL: 1350 dias
[OK] AMGN: 1350 dias
[OK] AXP: 1350 dias
[OK] BA: 1350 dias
[OK] CAT: 1350 dias
[OK] CRM: 1350 dias
[OK] CSCO: 1350 dias
[OK] CVX: 1350 dias
[OK] DIS: 1350 dias
[OK] DOW: 1350 dias
[OK] GS: 1350 dias
[OK] HD: 1350 dias
[OK] HON: 1350 dias
[OK] IBM: 1350 dias
[OK] INTC: 1350 dias
[OK] JNJ: 1350 dias
[OK] JPM: 1350 dias
[OK] KO: 1350 dias
[OK] MCD: 1350 dias
[OK] MMM: 1350 dias
[OK] MRK: 1350 dias
[OK] MSFT: 1350 dias
[OK] NKE: 1350 dias
[OK] PG: 1350 dias
[OK] TRV: 1350 dias
[OK] UNH: 1350 dias
[OK] V: 1350 dias
[OK] VZ: 1350 dias
[OK] WBA: 1350 dias
[OK] WMT: 1350 dias
[OK] summary -> exp_tecnn_mock_protocol/data/SUMMARY.csv
[searchae] iniciando busca TE-CNN-AAE
